In [ ]:
"""
Gera figura única 2x2 com legendas grandes para submissão AGU 2026.
Cole no Colab e rode tudo de uma vez.
"""

!pip install netCDF4 geopandas matplotlib -q

from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import geopandas as gpd
import requests, zipfile, glob, os

DATA_DIR   = "/content/drive/MyDrive/era5_brasil"
OUTPUT_DIR = "/content/drive/MyDrive/era5_brasil/figuras_en"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CULTURAS_PT = ["Lúpulo", "Pistache", "Avelã", "Amêndoa"]
NOMES_EN    = {
    "Lúpulo":  "Hops",
    "Pistache":"Pistachio",
    "Avelã":   "Hazelnut",
    "Amêndoa": "Almond",
}
LIMIAR  = 0.5
AREA_PX = 10978

# ── Shapefile ─────────────────────────────────────────────────────────────────
if not glob.glob("/tmp/estados_shp/*.shp"):
    r = requests.get(
        "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip",
        stream=True, timeout=120)
    with open("/tmp/estados.zip","wb") as f:
        for chunk in r.iter_content(65536*16): f.write(chunk)
    os.makedirs("/tmp/estados_shp/", exist_ok=True)
    with zipfile.ZipFile("/tmp/estados.zip") as z: z.extractall("/tmp/estados_shp/")
estados_all = gpd.read_file(glob.glob("/tmp/estados_shp/*.shp")[0])
estados = estados_all[estados_all["iso_a2"] == "BR"]
print(f"Shapefile OK: {len(estados)} states")

# ── Dados ─────────────────────────────────────────────────────────────────────
ds_emerg = xr.open_dataset(f"{DATA_DIR}/zonas_emergentes_filtradas.nc", engine="netcdf4")
lat = ds_emerg["lat"].values
lon = ds_emerg["lon"].values
print("Data loaded:", list(ds_emerg.data_vars))

# ── Cores e legenda ───────────────────────────────────────────────────────────
CORES  = ["#e8e4dc", "#1a6b1a", "#74b9e8", "#1a5fa8"]
LABELS = ["Unsuitable / excluded",
          "Currently suitable",
          "Emerging 2050",
          "Emerging 2080"]
cmap_e = mcolors.ListedColormap(CORES)
norm_e = mcolors.BoundaryNorm([0, 1, 2, 3, 4], 4)

# ── Figura 2x2 com tudo grande ────────────────────────────────────────────────
# Tamanho maior para legibilidade na impressão e no portal AGU
fig, axes = plt.subplots(2, 2, figsize=(20, 18))
axes = axes.flatten()

for i, c in enumerate(CULTURAS_PT):
    ax = axes[i]
    cl = ds_emerg[c].values

    ax.pcolormesh(lon, lat, cl, cmap=cmap_e, norm=norm_e,
                  rasterized=True, zorder=1)

    # Contorno dos estados
    estados.boundary.plot(ax=ax, linewidth=0.8, color="black", alpha=0.85)

    ax.set_xlim(-74, -28)
    ax.set_ylim(-34, 6)
    ax.set_facecolor("#d4dde8")

    # Título grande
    ax.set_title(NOMES_EN[c], fontsize=22, fontweight="bold", pad=10)

    # Eixos legíveis
    ax.set_xlabel("Longitude (deg)", fontsize=16)
    ax.set_ylabel("Latitude (deg)", fontsize=16)
    ax.tick_params(labelsize=14)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(10))

    # Caixa de estatísticas — fonte grande
    n1, n2, n3 = (int(np.nansum(cl==1)),
                  int(np.nansum(cl==2)),
                  int(np.nansum(cl==3)))
    ax.text(0.03, 0.04,
            f"Currently suitable: {n1 * AREA_PX / 1e6:.0f} Mha\n"
            f"Emerging 2050:      {n2 * AREA_PX / 1e6:.1f} Mha\n"
            f"Emerging 2080:      {n3 * AREA_PX / 1e6:.1f} Mha",
            transform=ax.transAxes, fontsize=14,
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                      edgecolor="gray", alpha=0.93), zorder=5)

    # Letra do painel
    ax.text(0.02, 0.97, f"({chr(97+i)})", transform=ax.transAxes,
            fontsize=18, fontweight="bold", va="top")

# ── Legenda global grande ─────────────────────────────────────────────────────
patches = [mpatches.Patch(color=c, label=l,
                           edgecolor="gray" if c=="#e8e4dc" else "none",
                           linewidth=0.8 if c=="#e8e4dc" else 0)
           for c, l in zip(CORES[1:], LABELS[1:])]

fig.legend(
    handles=patches,
    loc="lower center",
    ncol=3,
    fontsize=17,                      # fonte grande
    bbox_to_anchor=(0.5, -0.04),
    frameon=True,
    edgecolor="gray",
    title="Climate Suitability Index > 0.5  |  Land use filter: MapBiomas 2023",
    title_fontsize=15,
    handlelength=2.5,
    handleheight=1.8,
)

# ── Título geral ──────────────────────────────────────────────────────────────
plt.suptitle(
    "Current and Emerging Suitable Areas by Crop -- Brazil\n"
    "Envelope of SSP2-4.5 and SSP5-8.5  |  MIROC6 bias-corrected",
    fontsize=19, fontweight="bold", y=1.02
)

plt.tight_layout(w_pad=4, h_pad=4)

out = f"{OUTPUT_DIR}/fig_agu_emerging_zones.png"
plt.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nSaved: {out}")
print("Use this file as the AGU submission image.")

In [ ]:
"""
Gera as 3 figuras do artigo em inglês para submissão no AGU 2026.
Cole no Colab e rode tudo de uma vez.
"""

!pip install netCDF4 geopandas matplotlib -q

from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import geopandas as gpd
import requests, zipfile, glob, os

DATA_DIR   = "/content/drive/MyDrive/era5_brasil"
OUTPUT_DIR = "/content/drive/MyDrive/era5_brasil/figuras_en"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 4 culturas — nomes em inglês ──────────────────────────────────────────────
CULTURAS_PT = ["Lúpulo", "Pistache", "Avelã", "Amêndoa"]
NOMES_EN    = {
    "Lúpulo":  "Hops",
    "Pistache":"Pistachio",
    "Avelã":   "Hazelnut",
    "Amêndoa": "Almond",
}
LIMIAR  = 0.5
AREA_PX = 10978  # ha per pixel at 0.1 deg (~27S)

# ── Paleta ────────────────────────────────────────────────────────────────────
cmap_apt = mcolors.LinearSegmentedColormap.from_list(
    "suitability", ["#f5f0e8","#f0e68c","#90c44a","#1a6b1a"], N=256
)
norm_apt = mcolors.Normalize(vmin=0, vmax=1)

# ── Shapefile ─────────────────────────────────────────────────────────────────
if not glob.glob("/tmp/estados_shp/*.shp"):
    r = requests.get(
        "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip",
        stream=True, timeout=120)
    with open("/tmp/estados.zip","wb") as f:
        for chunk in r.iter_content(65536*16): f.write(chunk)
    os.makedirs("/tmp/estados_shp/", exist_ok=True)
    with zipfile.ZipFile("/tmp/estados.zip") as z: z.extractall("/tmp/estados_shp/")
estados_all = gpd.read_file(glob.glob("/tmp/estados_shp/*.shp")[0])
estados = estados_all[estados_all["iso_a2"] == "BR"]
print(f"Shapefile OK: {len(estados)} states")

# ── Carregar dados ────────────────────────────────────────────────────────────
ds_atual    = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_atual.nc",            engine="netcdf4")
ds_emerg    = xr.open_dataset(f"{DATA_DIR}/zonas_emergentes_filtradas.nc",        engine="netcdf4")
ds_245_2050 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp245_2041_2060.nc", engine="netcdf4")
ds_585_2050 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp585_2041_2060.nc", engine="netcdf4")
ds_245_2080 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp245_2071_2090.nc", engine="netcdf4")
ds_585_2080 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp585_2071_2090.nc", engine="netcdf4")

lat = ds_atual["lat"].values
lon = ds_atual["lon"].values
print("Data loaded:", list(ds_atual.data_vars))


# ── Função auxiliar ───────────────────────────────────────────────────────────
def formatar_eixo(ax, title, fontsize_title=14):
    estados.boundary.plot(ax=ax, linewidth=0.5, color="black", alpha=0.8)
    ax.set_xlim(-74, -28)
    ax.set_ylim(-34, 6)
    ax.set_facecolor("#d4dde8")
    ax.set_title(title, fontsize=fontsize_title, fontweight="bold", pad=6)
    ax.set_xlabel("Longitude (deg)", fontsize=11)
    ax.set_ylabel("Latitude (deg)", fontsize=11)
    ax.tick_params(labelsize=10)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(10))

def add_colorbar(im, ax, label, ticks=None):
    cb = plt.colorbar(im, ax=ax, shrink=0.82, pad=0.02, aspect=25)
    cb.set_label(label, fontsize=11)
    cb.ax.tick_params(labelsize=10)
    if ticks:
        cb.set_ticks(ticks)
    return cb


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Current suitability
# ═════════════════════════════════════════════════════════════════════════════
print("\nGenerating Figure 1...")
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

for i, c in enumerate(CULTURAS_PT):
    ax  = axes[i]
    apt = ds_atual[c].values

    im = ax.pcolormesh(lon, lat, apt, cmap=cmap_apt, norm=norm_apt,
                       rasterized=True, zorder=1)
    formatar_eixo(ax, NOMES_EN[c])

    mascara_br = ~np.isnan(apt)
    apt_v   = apt[mascara_br]
    n_total = int(mascara_br.sum())
    pct_a   = np.sum(apt_v > LIMIAR) / n_total * 100 if n_total > 0 else 0
    pct_o   = np.sum(apt_v > 0.7)    / n_total * 100 if n_total > 0 else 0
    mha_a   = np.sum(apt_v > LIMIAR) * AREA_PX / 1e6

    ax.text(0.03, 0.05,
            f"Suitable (CSI>0.5):  {pct_a:.1f}%  ({mha_a:.0f} Mha)\n"
            f"Optimal (CSI>0.7): {pct_o:.1f}%",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="gray", alpha=0.92), zorder=5)

    add_colorbar(im, ax, "Climate Suitability Index (0-1)",
                 ticks=[0, 0.25, 0.5, 0.75, 1.0])

    ax.text(0.02, 0.97, f"({chr(97+i)})", transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top")

plt.suptitle(
    "Current Climate Suitability -- Brazil (1991-2020)\n"
    "Land use filter: MapBiomas Collection 9 (2023)",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout(w_pad=3, h_pad=3)
out1 = f"{OUTPUT_DIR}/fig1_current_suitability.png"
plt.savefig(out1, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out1}")


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Suitability change by scenario
# ═════════════════════════════════════════════════════════════════════════════
print("\nGenerating Figure 2...")
fig, axes = plt.subplots(4, 2, figsize=(14, 22))

cmap_d = plt.cm.RdBu_r
norm_d = mcolors.TwoSlopeNorm(vmin=-0.35, vcenter=0, vmax=0.35)

cenarios_plot = [
    (ds_245_2050, "SSP2-4.5 -- 2041-2060"),
    (ds_585_2080, "SSP5-8.5 -- 2071-2090"),
]

for row, c in enumerate(CULTURAS_PT):
    apt_atual = ds_atual[c].values
    for col, (ds_fut, label) in enumerate(cenarios_plot):
        ax      = axes[row, col]
        apt_fut = ds_fut[c].values
        delta   = apt_fut - apt_atual

        im = ax.pcolormesh(lon, lat, delta, cmap=cmap_d, norm=norm_d,
                           rasterized=True, zorder=1)
        formatar_eixo(ax, f"{NOMES_EN[c]} -- {label}", fontsize_title=12)

        if col == 0:
            ax.set_ylabel(f"{NOMES_EN[c]}\nLatitude (deg)",
                          fontsize=11, fontweight="bold")

        ganho = int(np.nansum((apt_fut > LIMIAR) & (apt_atual <= LIMIAR)))
        perda = int(np.nansum((apt_fut <= LIMIAR) & (apt_atual > LIMIAR)))
        ax.text(0.03, 0.05,
                f"+{ganho} px gain  /  -{perda} px loss",
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor="gray", alpha=0.9), zorder=5)

        add_colorbar(im, ax, "Delta CSI",
                     ticks=[-0.3, -0.15, 0, 0.15, 0.3])

        ax.text(0.02, 0.97, f"({chr(97 + row*2 + col)})",
                transform=ax.transAxes, fontsize=13,
                fontweight="bold", va="top")

plt.suptitle(
    "Climate Suitability Change by Scenario\n"
    "MIROC6 bias-corrected (delta method) -- MapBiomas land use filter 2023",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout(w_pad=3, h_pad=3)
out2 = f"{OUTPUT_DIR}/fig2_suitability_change.png"
plt.savefig(out2, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out2}")


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Emerging zones (AGU submission figure)
# ═════════════════════════════════════════════════════════════════════════════
print("\nGenerating Figure 3 (AGU submission)...")
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

CORES  = ["#e8e4dc", "#1a6b1a", "#74b9e8", "#1a5fa8"]
LABELS = ["Unsuitable / excluded", "Currently suitable",
          "Emerging 2050", "Emerging 2080"]
cmap_e = mcolors.ListedColormap(CORES)
norm_e = mcolors.BoundaryNorm([0, 1, 2, 3, 4], 4)

for i, c in enumerate(CULTURAS_PT):
    ax = axes[i]
    cl = ds_emerg[c].values

    ax.pcolormesh(lon, lat, cl, cmap=cmap_e, norm=norm_e,
                  rasterized=True, zorder=1)
    formatar_eixo(ax, NOMES_EN[c])

    n1, n2, n3 = (int(np.nansum(cl==1)),
                  int(np.nansum(cl==2)),
                  int(np.nansum(cl==3)))

    ax.text(0.03, 0.05,
            f"Currently suitable: {n1 * AREA_PX / 1e6:.0f} Mha\n"
            f"Emerging 2050:      {n2 * AREA_PX / 1e6:.1f} Mha\n"
            f"Emerging 2080:      {n3 * AREA_PX / 1e6:.1f} Mha",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="gray", alpha=0.92), zorder=5)

    ax.text(0.02, 0.97, f"({chr(97+i)})", transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top")

patches = [mpatches.Patch(color=c, label=l,
                           edgecolor="gray" if c=="#e8e4dc" else "none",
                           linewidth=0.5 if c=="#e8e4dc" else 0)
           for c, l in zip(CORES[1:], LABELS[1:])]
fig.legend(handles=patches, loc="lower center", ncol=3,
           fontsize=12, bbox_to_anchor=(0.5, -0.03),
           frameon=True, edgecolor="gray",
           title="Climate Suitability Index > 0.5 -- land use filter applied",
           title_fontsize=11)

plt.suptitle(
    "Current and Emerging Suitable Areas by Crop -- Brazil\n"
    "Envelope of SSP2-4.5 and SSP5-8.5 -- MapBiomas Collection 9 (2023)",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout(w_pad=3, h_pad=3)
out3 = f"{OUTPUT_DIR}/fig3_emerging_zones.png"
plt.savefig(out3, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out3}")

print(f"\nAll figures saved in: {OUTPUT_DIR}")
print("Recommended AGU submission image: fig3_emerging_zones.png")

In [ ]:
!pip install netCDF4 -q

from google.colab import drive
drive.mount('/content/drive')

import xarray as xr

DATA_DIR = "/content/drive/MyDrive/era5_brasil"

ds_atual = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_atual.nc", engine="netcdf4")
ds_emerg = xr.open_dataset(f"{DATA_DIR}/zonas_emergentes_filtradas.nc", engine="netcdf4")

print("aptidao_filtrada_atual:", list(ds_atual.data_vars))
print("zonas_emergentes_filtradas:", list(ds_emerg.data_vars))

In [ ]:
!pip install netCDF4 geopandas matplotlib -q

from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import geopandas as gpd
import requests, zipfile, glob, os

DATA_DIR   = "/content/drive/MyDrive/era5_brasil"
OUTPUT_DIR = "/content/drive/MyDrive/era5_brasil/figuras_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 4 culturas com nomes exatos dos arquivos ──────────────────────────────────
CULTURAS = ["Lúpulo", "Pistache", "Avelã", "Amêndoa"]
LIMIAR   = 0.5
AREA_PX  = 10978

# ── Paleta sem vermelho ───────────────────────────────────────────────────────
cmap_apt = mcolors.LinearSegmentedColormap.from_list(
    "aptidao", ["#f5f0e8","#f0e68c","#90c44a","#1a6b1a"], N=256
)
norm_apt = mcolors.Normalize(vmin=0, vmax=1)

# ── Shapefile estados ────────────────────────────────────────────────────────
import geopandas as gpd

# Natural Earth — servidor mais estável que o IBGE
url_ne = "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"

if not glob.glob("/tmp/estados_shp/*.shp"):
    print("Baixando shapefile...")
    r = requests.get(url_ne, stream=True, timeout=120)
    with open("/tmp/estados.zip","wb") as f:
        for chunk in r.iter_content(65536*16): f.write(chunk)
    os.makedirs("/tmp/estados_shp/", exist_ok=True)
    with zipfile.ZipFile("/tmp/estados.zip") as z: z.extractall("/tmp/estados_shp/")

# Filtrar só estados brasileiros
estados_all = gpd.read_file(glob.glob("/tmp/estados_shp/*.shp")[0])
estados = estados_all[estados_all["iso_a2"] == "BR"]
print(f"✓ {len(estados)} estados brasileiros carregados")

# ── Carregar dados ────────────────────────────────────────────────────────────
ds_atual    = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_atual.nc",            engine="netcdf4")
ds_emerg    = xr.open_dataset(f"{DATA_DIR}/zonas_emergentes_filtradas.nc",        engine="netcdf4")
ds_245_2050 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp245_2041_2060.nc", engine="netcdf4")
ds_585_2050 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp585_2041_2060.nc", engine="netcdf4")
ds_245_2080 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp245_2071_2090.nc", engine="netcdf4")
ds_585_2080 = xr.open_dataset(f"{DATA_DIR}/aptidao_filtrada_ssp585_2071_2090.nc", engine="netcdf4")

lat = ds_atual["lat"].values
lon = ds_atual["lon"].values
print("✓ Dados carregados:", list(ds_atual.data_vars))

# ── Funções auxiliares ────────────────────────────────────────────────────────
def formatar_eixo(ax, title, fontsize_title=14):
    estados.boundary.plot(ax=ax, linewidth=0.5, color="black", alpha=0.8)
    ax.set_xlim(-74, -28)
    ax.set_ylim(-34, 6)
    ax.set_facecolor("#d4dde8")
    ax.set_title(title, fontsize=fontsize_title, fontweight="bold", pad=6)
    ax.set_xlabel("Longitude (°)", fontsize=11)
    ax.set_ylabel("Latitude (°)", fontsize=11)
    ax.tick_params(labelsize=10)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(10))

def add_colorbar(im, ax, label, ticks=None):
    cb = plt.colorbar(im, ax=ax, shrink=0.82, pad=0.02, aspect=25)
    cb.set_label(label, fontsize=11)
    cb.ax.tick_params(labelsize=10)
    if ticks: cb.set_ticks(ticks)
    return cb

# ═════════════════════════════════════════════════════════════════════════════
# FIGURA 1 — Aptidão atual
# ═════════════════════════════════════════════════════════════════════════════
print("\nGerando Figura 1...")
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

for i, c in enumerate(CULTURAS):
    ax  = axes[i]
    apt = ds_atual[c].values

    im = ax.pcolormesh(lon, lat, apt, cmap=cmap_apt, norm=norm_apt,
                       rasterized=True, zorder=1)
    formatar_eixo(ax, c)

    mascara_br = ~np.isnan(apt)
    apt_v   = apt[mascara_br]
    n_total = int(mascara_br.sum())
    pct_a   = np.sum(apt_v > LIMIAR) / n_total * 100 if n_total > 0 else 0
    pct_o   = np.sum(apt_v > 0.7)    / n_total * 100 if n_total > 0 else 0
    mha_a   = np.sum(apt_v > LIMIAR) * AREA_PX / 1e6

    ax.text(0.03, 0.05,
            f"Apta (IAC>0,5):  {pct_a:.1f}%  ({mha_a:.0f} Mha)\n"
            f"Ótima (IAC>0,7): {pct_o:.1f}%",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="gray", alpha=0.92), zorder=5)

    add_colorbar(im, ax, "Índice de Aptidão Climática (0–1)",
                 ticks=[0, 0.25, 0.5, 0.75, 1.0])

    ax.text(0.02, 0.97, f"({chr(97+i)})", transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top")

plt.suptitle(
    "Aptidão Climática Atual — Brasil (1991–2020)\n"
    "Filtro: MapBiomas Coleção 9 (2023) — uso agrícola",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout(w_pad=3, h_pad=3)
out1 = f"{OUTPUT_DIR}/fig1_aptidao_atual_v2.png"
plt.savefig(out1, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✓ {out1}")

# ═════════════════════════════════════════════════════════════════════════════
# FIGURA 2 — Delta de aptidão
# ═════════════════════════════════════════════════════════════════════════════
print("\nGerando Figura 2...")
fig, axes = plt.subplots(4, 2, figsize=(14, 22))

cmap_d = plt.cm.RdBu_r
norm_d = mcolors.TwoSlopeNorm(vmin=-0.35, vcenter=0, vmax=0.35)

cenarios_plot = [
    (ds_245_2050, "SSP2-4.5 · 2041–2060"),
    (ds_585_2080, "SSP5-8.5 · 2071–2090"),
]

for row, c in enumerate(CULTURAS):
    apt_atual = ds_atual[c].values
    for col, (ds_fut, label) in enumerate(cenarios_plot):
        ax      = axes[row, col]
        apt_fut = ds_fut[c].values
        delta   = apt_fut - apt_atual

        im = ax.pcolormesh(lon, lat, delta, cmap=cmap_d, norm=norm_d,
                           rasterized=True, zorder=1)
        formatar_eixo(ax, f"{c} — {label}", fontsize_title=12)

        if col == 0:
            ax.set_ylabel(f"{c}\nLatitude (°)", fontsize=11, fontweight="bold")

        ganho = int(np.nansum((apt_fut > LIMIAR) & (apt_atual <= LIMIAR)))
        perda = int(np.nansum((apt_fut <= LIMIAR) & (apt_atual > LIMIAR)))
        ax.text(0.03, 0.05,
                f"+{ganho} px ganho  /  −{perda} px perda",
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor="gray", alpha=0.9), zorder=5)

        add_colorbar(im, ax, "ΔAptidão",
                     ticks=[-0.3, -0.15, 0, 0.15, 0.3])

        ax.text(0.02, 0.97, f"({chr(97 + row*2 + col)})",
                transform=ax.transAxes, fontsize=13, fontweight="bold", va="top")

plt.suptitle(
    "Mudança de Aptidão Climática por Cenário\n"
    "MIROC6 bias-corrected (delta method) · Filtro MapBiomas 2023",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout(w_pad=3, h_pad=3)
out2 = f"{OUTPUT_DIR}/fig2_delta_aptidao_v2.png"
plt.savefig(out2, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✓ {out2}")

# ═════════════════════════════════════════════════════════════════════════════
# FIGURA 3 — Regiões emergentes
# ═════════════════════════════════════════════════════════════════════════════
print("\nGerando Figura 3...")
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

CORES  = ["#e8e4dc", "#1a6b1a", "#74b9e8", "#1a5fa8"]
LABELS = ["Inapta / excluída", "Apta atual", "Emergente 2050", "Emergente 2080"]
cmap_e = mcolors.ListedColormap(CORES)
norm_e = mcolors.BoundaryNorm([0, 1, 2, 3, 4], 4)

for i, c in enumerate(CULTURAS):
    ax = axes[i]
    cl = ds_emerg[c].values

    ax.pcolormesh(lon, lat, cl, cmap=cmap_e, norm=norm_e, rasterized=True, zorder=1)
    formatar_eixo(ax, c)

    n1, n2, n3 = int(np.nansum(cl==1)), int(np.nansum(cl==2)), int(np.nansum(cl==3))

    ax.text(0.03, 0.05,
            f"Apta atual:      {n1 * AREA_PX / 1e6:.0f} Mha\n"
            f"Emergente 2050:  {n2 * AREA_PX / 1e6:.1f} Mha\n"
            f"Emergente 2080:  {n3 * AREA_PX / 1e6:.1f} Mha",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="gray", alpha=0.92), zorder=5)

    ax.text(0.02, 0.97, f"({chr(97+i)})", transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top")

patches = [mpatches.Patch(color=c, label=l,
                           edgecolor="gray" if c=="#e8e4dc" else "none",
                           linewidth=0.5 if c=="#e8e4dc" else 0)
           for c, l in zip(CORES[1:], LABELS[1:])]
fig.legend(handles=patches, loc="lower center", ncol=3,
           fontsize=12, bbox_to_anchor=(0.5, -0.03),
           frameon=True, edgecolor="gray",
           title="Classificação de aptidão (IAC > 0,5)",
           title_fontsize=12)

plt.suptitle(
    "Regiões Aptas e Emergentes por Cultura — Brasil\n"
    "Envelope SSP2-4.5 e SSP5-8.5 · Filtro MapBiomas Coleção 9 (2023)",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout(w_pad=3, h_pad=3)
out3 = f"{OUTPUT_DIR}/fig3_emergentes_v2.png"
plt.savefig(out3, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✓ {out3}")

print(f"\n✓ Três figuras salvas em: {OUTPUT_DIR}")

In [ ]:
!pip install netCDF4 -q
import IPython
IPython.Application.instance().kernel.do_shutdown(True)  # reinicia kernel automaticamente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install requests xarray netCDF4 -q

import requests
import time
import os

KEY = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"

headers = {
    "PRIVATE-TOKEN": KEY,
    "Content-Type": "application/json"
}

def submit_request(payload):
    r = requests.post(
        f"{BASE}/retrieve/v1/processes/sis-agrometeorological-indicators/execution",
        json={"inputs": payload},
        headers=headers
    )
    r.raise_for_status()
    return r.json()["jobID"]

def wait_and_download(job_id, output_path, timeout=600):
    for _ in range(timeout // 10):
        r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}", headers=headers)
        status = r.json().get("status")
        print(f"  status: {status}")
        if status == "successful":
            # pegar URL de download
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            url = r2.json()["asset"]["value"]["href"]
            # baixar arquivo
            r3 = requests.get(url, stream=True)
            with open(output_path, "wb") as f:
                for chunk in r3.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        elif status == "failed":
            print(f"  ✗ Job falhou: {r.json()}")
            return False
        time.sleep(10)
    print("  ✗ Timeout")
    return False

# Teste com um bloco pequeno
payload = {
    "variable": "2m_temperature_24_hour_maximum",
    "year": ["1991"],
    "month": ["01"],
    "day": ["01", "02", "03"],
    "area": [6, -74, 1, -69],
    "data_format": "netcdf",
    "version": "2_0",
}

print("Submetendo request...")
job_id = submit_request(payload)
print(f"Job ID: {job_id}")

ok = wait_and_download(job_id, "teste_rest.nc")
if ok:
    print("✓ Arquivo baixado com sucesso!")
    import xarray as xr
    ds = xr.open_dataset("teste_rest.nc")
    print(ds)

In [ ]:
# Ver o erro detalhado do job
r = requests.get(
    f"{BASE}/retrieve/v1/jobs/95243206-d2ca-4bc1-ac25-5325812af18e/results",
    headers=headers
)
print(r.status_code)
print(r.json())

In [ ]:
# Ver variáveis disponíveis no dataset
r = requests.get(
    f"{BASE}/retrieve/v1/processes/sis-agrometeorological-indicators",
    headers=headers
)
import json
info = r.json()

# Pegar os inputs disponíveis
for inp in info.get("inputs", {}).items():
    print(inp[0], ":", list(inp[1].get("schema", {}).get("enum", [inp[1]]))[:5])

In [ ]:
r = requests.get(
    f"{BASE}/retrieve/v1/processes/sis-agrometeorological-indicators",
    headers=headers
)
import json
print(json.dumps(r.json(), indent=2)[:3000])  # primeiros 3000 chars

In [ ]:
payload = {
    "variable": "2m_temperature",
    "statistic": ["24_hour_maximum"],
    "year": ["1991"],
    "month": ["01"],
    "day": ["01", "02", "03"],
    "area": [6, -74, 1, -69],
    "data_format": "netcdf",
    "version": "2_0",
}

print("Submetendo...")
job_id = submit_request(payload)
print(f"Job ID: {job_id}")

ok = wait_and_download(job_id, "teste_v2_correto.nc")
if ok:
    import xarray as xr
    ds = xr.open_dataset("teste_v2_correto.nc")
    print("✓ Funcionou!")
    print(ds)

In [ ]:
!pip install netCDF4 -q

import xarray as xr
ds = xr.open_dataset("teste_v2_correto.nc", engine="netcdf4")
print("✓ Funcionou!")
print(ds)

In [ ]:
import os

# Ver tamanho e tipo do arquivo
print("Tamanho:", os.path.getsize("teste_v2_correto.nc"), "bytes")

# Ver os primeiros bytes para identificar o formato
with open("teste_v2_correto.nc", "rb") as f:
    header = f.read(10)
print("Header (hex):", header.hex())
print("Header (ascii):", header)

In [ ]:
import zipfile
import xarray as xr
import os

# Descompactar
with zipfile.ZipFile("teste_v2_correto.nc", "r") as z:
    print("Arquivos dentro do ZIP:", z.namelist())
    z.extractall("agera5_extraido/")

# Abrir o NetCDF extraído
arquivos_nc = [f for f in os.listdir("agera5_extraido/") if f.endswith(".nc")]
print("NetCDFs encontrados:", arquivos_nc)

ds = xr.open_dataset(f"agera5_extraido/{arquivos_nc[0]}", engine="netcdf4")
print("✓ Funcionou!")
print(ds)

In [ ]:
import requests, zipfile, xarray as xr, os, time
from itertools import product
from google.colab import drive
drive.mount('/content/drive')

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

OUTPUT_DIR = "/content/drive/MyDrive/agera5_brasil"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ANOS  = [str(a) for a in range(1991, 2021)]
MESES = [f"{m:02d}" for m in range(1, 13)]
DIAS  = [f"{d:02d}" for d in range(1, 32)]

VARIAVEIS = {
    "tmax":   ("2m_temperature",          ["24_hour_maximum"]),
    "tmin":   ("2m_temperature",          ["24_hour_minimum"]),
    "tmean":  ("2m_temperature",          ["24_hour_mean"]),
    "precip": ("precipitation_flux",      None),
    "etp":    ("reference_evapotranspiration", None),
}

BLOCOS_LAT = [(6,1),(1,-4),(-4,-9),(-9,-14),(-14,-19),(-19,-24),(-24,-29),(-29,-34)]
BLOCOS_LON = [(-74,-69),(-69,-64),(-64,-59),(-59,-54),(-54,-49),
              (-49,-44),(-44,-39),(-39,-34),(-34,-29),(-29,-28)]

def submit(payload):
    r = requests.post(
        f"{BASE}/retrieve/v1/processes/sis-agrometeorological-indicators/execution",
        json={"inputs": payload}, headers=headers)
    r.raise_for_status()
    return r.json()["jobID"]

def wait_download(job_id, path, timeout=600):
    for _ in range(timeout // 10):
        r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}", headers=headers)
        status = r.json().get("status")
        if status == "successful":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            url = r2.json()["asset"]["value"]["href"]
            r3 = requests.get(url, stream=True)
            with open(path, "wb") as f:
                for chunk in r3.iter_content(8192): f.write(chunk)
            return True
        elif status == "failed":
            print(f"    ✗ falhou")
            return False
        time.sleep(10)
    return False

conta = 0
total = len(VARIAVEIS) * len(BLOCOS_LAT) * len(BLOCOS_LON) * len(ANOS) * len(MESES)

for var_key, (var_name, statistic) in VARIAVEIS.items():
    for (north, south), (west, east) in product(BLOCOS_LAT, BLOCOS_LON):
        for ano in ANOS:
            # arquivo final concatenado por ano-bloco
            nc_final = os.path.join(
                OUTPUT_DIR, var_key,
                f"N{north}_S{south}_W{west}_E{east}_{ano}.nc".replace("-","m")
            )
            os.makedirs(os.path.dirname(nc_final), exist_ok=True)

            if os.path.exists(nc_final):
                print(f"✓ Já existe: {var_key} {ano} bloco N{north}")
                continue

            meses_nc = []  # arquivos mensais para concatenar

            for mes in MESES:
                conta += 1
                tmp_zip = f"/tmp/agera5_{var_key}_{ano}_{mes}.zip"
                tmp_dir = f"/tmp/agera5_{var_key}_{ano}_{mes}/"

                if not os.path.exists(tmp_dir):
                    payload = {
                        "variable": var_name,
                        "year": [ano],
                        "month": [mes],
                        "day": DIAS,
                        "area": [north, west, south, east],
                        "data_format": "netcdf",
                        "version": "2_0",
                    }
                    if statistic:
                        payload["statistic"] = statistic

                    print(f"[{conta}/{total}] ↓ {var_key} {ano}-{mes} N{north} W{west}")
                    job_id = submit(payload)
                    ok = wait_download(job_id, tmp_zip)

                    if not ok:
                        continue

                    os.makedirs(tmp_dir, exist_ok=True)
                    with zipfile.ZipFile(tmp_zip, "r") as z:
                        z.extractall(tmp_dir)
                    os.remove(tmp_zip)

                # concatenar dias do mês
                ncs = sorted([os.path.join(tmp_dir, f)
                               for f in os.listdir(tmp_dir) if f.endswith(".nc")])
                if ncs:
                    ds_mes = xr.open_mfdataset(ncs, combine="by_coords")
                    meses_nc.append(ds_mes)

            # concatenar 12 meses e salvar
            if meses_nc:
                ds_ano = xr.concat(meses_nc, dim="time")
                ds_ano.to_netcdf(nc_final)
                print(f"  ✓ Salvo: {nc_final}")

                # limpar temporários
                import shutil
                for mes in MESES:
                    tmp_dir = f"/tmp/agera5_{var_key}_{ano}_{mes}/"
                    if os.path.exists(tmp_dir):
                        shutil.rmtree(tmp_dir)

print("\n✓ Download completo!")

In [ ]:
import requests, os, zipfile

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

# ERA5 monthly averaged — cobre Brasil inteiro sem limite de área
# Dataset diferente: reanalysis-era5-land-monthly-means

def submit(payload, dataset="reanalysis-era5-land-monthly-means"):
    r = requests.post(
        f"{BASE}/retrieve/v1/processes/{dataset}/execution",
        json={"inputs": payload}, headers=headers)
    print("Status submit:", r.status_code)
    print(r.json())
    return r.json().get("jobID")

# Teste — temperatura 2m mensal, Brasil inteiro, 1991-2020
payload = {
    "variable": "2m_temperature",
    "product_type": "monthly_averaged_reanalysis",
    "year": [str(a) for a in range(1991, 2021)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "time": "00:00",
    "area": [6, -74, -34, -28],   # Brasil inteiro de uma vez!
    "data_format": "netcdf",
}

job_id = submit(payload)
print("Job ID:", job_id)

In [ ]:
import requests, time, zipfile, xarray as xr

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

def wait_download(job_id, path, timeout=1800):
    for _ in range(timeout // 15):
        r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}", headers=headers)
        status = r.json().get("status")
        print(f"  status: {status}")
        if status == "successful":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            url = r2.json()["asset"]["value"]["href"]
            r3 = requests.get(url, stream=True)
            with open(path, "wb") as f:
                for chunk in r3.iter_content(8192): f.write(chunk)
            print(f"  ✓ Baixado: {path}")
            return True
        elif status == "failed":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            print(f"  ✗ Falhou:", r2.json())
            return False
        time.sleep(15)
    return False

ok = wait_download("e1b68d45-8140-41fb-99e1-281b1d41a0e1", "era5_t2m_mensal.nc")

if ok:
    with open("era5_t2m_mensal.nc", "rb") as f:
        print("Header:", f.read(4).hex())

In [ ]:
import requests, time

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

VARIAVEIS = {
    "t2m_min":  "2m_temperature",           # vamos pedir min separado
    "precip":   "total_precipitation",
    "et":       "potential_evaporation",
    "dewpoint": "2m_dewpoint_temperature",  # para calcular ETP se necessário
}

BASE_PAYLOAD = {
    "product_type": "monthly_averaged_reanalysis",
    "year": [str(a) for a in range(1991, 2021)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "time": "00:00",
    "area": [6, -74, -34, -28],
    "data_format": "netcdf",
}

jobs = {}
for var_key, var_name in VARIAVEIS.items():
    payload = {**BASE_PAYLOAD, "variable": var_name}
    r = requests.post(
        f"{BASE}/retrieve/v1/processes/reanalysis-era5-land-monthly-means/execution",
        json={"inputs": payload}, headers=headers)
    job_id = r.json().get("jobID")
    jobs[var_key] = job_id
    print(f"✓ {var_key}: job {job_id}")

print("\nJobs submetidos:", jobs)

In [ ]:
import requests, time, zipfile, os, shutil
from google.colab import drive
drive.mount('/content/drive')

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

OUTPUT_DIR = "/content/drive/MyDrive/era5_brasil"
os.makedirs(OUTPUT_DIR, exist_ok=True)

jobs = {
    't2m_min':  'fd2d1790-e7c4-4bc7-b8b9-4cdded5a7c36',
    'precip':   '7f181886-dd06-47ec-82e3-725ea83bee5c',
    'et':       '77f6b3e3-d20f-460f-89e2-6290cbd6ed4c',
    'dewpoint': '444139a5-61cf-425c-bf4d-0d3951f6c119',
}

def wait_download(var_key, job_id, timeout=1800):
    print(f"\n⏳ Aguardando {var_key}...")
    for _ in range(timeout // 15):
        r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}", headers=headers)
        status = r.json().get("status")
        print(f"  {var_key}: {status}")
        if status == "successful":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            url = r2.json()["asset"]["value"]["href"]
            zip_path = f"/tmp/{var_key}.zip"
            r3 = requests.get(url, stream=True)
            with open(zip_path, "wb") as f:
                for chunk in r3.iter_content(8192): f.write(chunk)
            with zipfile.ZipFile(zip_path, "r") as z:
                names = z.namelist()
                z.extractall(f"/tmp/{var_key}/")
            nc_src = f"/tmp/{var_key}/{names[0]}"
            nc_dst = os.path.join(OUTPUT_DIR, f"brasil_{var_key}_1991_2020.nc")
            shutil.copy2(nc_src, nc_dst)  # ← corrigido
            os.remove(zip_path)
            os.remove(nc_src)
            print(f"  ✓ Salvo: {nc_dst}")
            return True
        elif status == "failed":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            print(f"  ✗ Falhou: {r2.json()}")
            return False
        time.sleep(15)
    print(f"  ✗ Timeout: {var_key}")
    return False

for var_key, job_id in jobs.items():
    wait_download(var_key, job_id)

print("\n✓ Todos os downloads concluídos!")

In [ ]:
# Submeter Tmax
payload_tmax = {
    "variable": "maximum_2m_temperature_in_the_last_24_hours",
    "product_type": "monthly_averaged_reanalysis",
    "year": [str(a) for a in range(1991, 2021)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "time": "00:00",
    "area": [6, -74, -34, -28],
    "data_format": "netcdf",
}

r = requests.post(
    f"{BASE}/retrieve/v1/processes/reanalysis-era5-land-monthly-means/execution",
    json={"inputs": payload_tmax}, headers=headers)

print("Status:", r.status_code)
job_tmax = r.json().get("jobID")
print("Job ID tmax:", job_tmax)

# Aguardar e baixar
wait_download("t2m_max", job_tmax)

# Verificar todos os arquivos no Drive
import os
files = os.listdir("/content/drive/MyDrive/era5_brasil")
print("\nArquivos no Drive:")
for f in sorted(files):
    size = os.path.getsize(f"/content/drive/MyDrive/era5_brasil/{f}") / 1e6
    print(f"  {f} — {size:.1f} MB")

In [ ]:
 # Ver estrutura completa do campo variable
import json
print(json.dumps(inputs["variable"], indent=2)[:3000])

In [ ]:
# Verificar se AgERA5 tem statistic mensal
r = requests.get(
    f"{BASE}/retrieve/v1/processes/sis-agrometeorological-indicators",
    headers=headers)

inputs = r.json()["inputs"]
print("Campos disponíveis:", list(inputs.keys()))

# Ver se tem campo de agregação temporal
for key in inputs:
    print(f"\n{key}:", list(inputs[key].get("schema", {}).get("items", {}).get("enum", [inputs[key]]))[:5])

In [ ]:
!pip install netCDF4 -q

import xarray as xr, os

DIR = "/content/drive/MyDrive/era5_brasil"

# Ver o que existe no Drive
print("Arquivos disponíveis:")
for f in sorted(os.listdir(DIR)):
    size = os.path.getsize(f"{DIR}/{f}") / 1e6
    print(f"  {f} — {size:.1f} MB")

In [ ]:
import requests

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

BASE_PAYLOAD = {
    "product_type": "monthly_averaged_reanalysis",
    "year": [str(a) for a in range(1991, 2021)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "time": "00:00",
    "area": [6, -74, -34, -28],
    "data_format": "netcdf",
}

jobs = {}

# Tmean
r = requests.post(
    f"{BASE}/retrieve/v1/processes/reanalysis-era5-land-monthly-means/execution",
    json={"inputs": {**BASE_PAYLOAD, "variable": "2m_temperature"}},
    headers=headers)
jobs["t2m_mean"] = r.json().get("jobID")
print(f"✓ t2m_mean: {jobs['t2m_mean']}")

# Tmin de novo (por garantia — já temos mas vamos confirmar)
# Tmax via skin_temperature como proxy
r = requests.post(
    f"{BASE}/retrieve/v1/processes/reanalysis-era5-land-monthly-means/execution",
    json={"inputs": {**BASE_PAYLOAD, "variable": "skin_temperature"}},
    headers=headers)
jobs["skin_temp"] = r.json().get("jobID")
print(f"✓ skin_temp: {jobs['skin_temp']}")

print("\nJobs:", jobs)

In [ ]:
import requests, time, zipfile, os, shutil
from google.colab import drive
drive.mount('/content/drive')

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

OUTPUT_DIR = "/content/drive/MyDrive/era5_brasil"
os.makedirs(OUTPUT_DIR, exist_ok=True)

jobs = {
    't2m_mean': '2ae23bc4-cf48-4db5-b206-e567eb67a981',
    'skin_temp': 'c5aa6b32-0616-424f-8b0b-36ebcb4fa529',
}

def wait_download(var_key, job_id, timeout=1800):
    print(f"\n⏳ Aguardando {var_key}...")
    for _ in range(timeout // 15):
        r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}", headers=headers)
        status = r.json().get("status")
        print(f"  {var_key}: {status}")
        if status == "successful":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            url = r2.json()["asset"]["value"]["href"]
            zip_path = f"/tmp/{var_key}.zip"
            r3 = requests.get(url, stream=True)
            with open(zip_path, "wb") as f:
                for chunk in r3.iter_content(8192): f.write(chunk)
            with zipfile.ZipFile(zip_path, "r") as z:
                names = z.namelist()
                z.extractall(f"/tmp/{var_key}/")
            nc_src = f"/tmp/{var_key}/{names[0]}"
            nc_dst = os.path.join(OUTPUT_DIR, f"brasil_{var_key}_1991_2020.nc")
            shutil.copy2(nc_src, nc_dst)
            os.remove(zip_path)
            os.remove(nc_src)
            print(f"  ✓ Salvo: {nc_dst}")
            return True
        elif status == "failed":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            print(f"  ✗ Falhou: {r2.json()}")
            return False
        time.sleep(15)
    print(f"  ✗ Timeout: {var_key}")
    return False

for var_key, job_id in jobs.items():
    wait_download(var_key, job_id)

# Verificar tudo
print("\nArquivos no Drive:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1e6
    print(f"  {f} — {size:.1f} MB")

In [ ]:
import xarray as xr
import numpy as np
import os

DIR = "/content/drive/MyDrive/era5_brasil"

# Abrir Tmean e Tmin
ds_mean = xr.open_dataset(f"{DIR}/brasil_t2m_mean_1991_2020.nc", engine="netcdf4")
ds_min  = xr.open_dataset(f"{DIR}/brasil_t2m_min_1991_2020.nc",  engine="netcdf4")

print("Tmean coords:", list(ds_mean.coords))
print("Tmin coords:",  list(ds_min.coords))
print("Tmean var:", list(ds_mean.data_vars))
print("Tmin var:",  list(ds_min.data_vars))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import os

DIR = "/content/drive/MyDrive/era5_brasil"

ds_mean = xr.open_dataset(f"{DIR}/brasil_t2m_mean_1991_2020.nc", engine="netcdf4")
ds_min  = xr.open_dataset(f"{DIR}/brasil_t2m_min_1991_2020.nc",  engine="netcdf4")

print("Tmean coords:", list(ds_mean.coords))
print("Tmin coords:",  list(ds_min.coords))
print("Tmean var:", list(ds_mean.data_vars))
print("Tmin var:",  list(ds_min.data_vars))

In [ ]:
import numpy as np

# Carregar variáveis em Kelvin
tmean = ds_mean['t2m']
tmin  = ds_min['t2m']

# Tmax estimado (Allen et al. 1998, FAO-56)
tmax = 2 * tmean - tmin

# Converter tudo para Celsius
tmean_c = tmean - 273.15
tmin_c  = tmin  - 273.15
tmax_c  = tmax  - 273.15

# Verificar ranges
print("Tmean (°C): min", round(float(tmean_c.min()), 1), "max", round(float(tmean_c.max()), 1))
print("Tmin  (°C): min", round(float(tmin_c.min()),  1), "max", round(float(tmin_c.max()),  1))
print("Tmax  (°C): min", round(float(tmax_c.min()),  1), "max", round(float(tmax_c.max()),  1))

# Montar dataset final com todas as variáveis
ds_precip = xr.open_dataset(f"{DIR}/brasil_precip_1991_2020.nc",   engine="netcdf4")
ds_et     = xr.open_dataset(f"{DIR}/brasil_et_1991_2020.nc",       engine="netcdf4")
ds_dew    = xr.open_dataset(f"{DIR}/brasil_dewpoint_1991_2020.nc", engine="netcdf4")

print("\nPrecip var:",   list(ds_precip.data_vars))
print("ET var:",        list(ds_et.data_vars))
print("Dewpoint var:",  list(ds_dew.data_vars))
print("\nPrecip coords:", list(ds_precip.coords))

In [ ]:
# Ver os metadados internos de cada arquivo para confirmar o que é cada um
import xarray as xr

DIR = "/content/drive/MyDrive/era5_brasil"

for var_key in ['t2m_mean', 't2m_min']:
    ds = xr.open_dataset(f"{DIR}/brasil_{var_key}_1991_2020.nc", engine="netcdf4")
    print(f"\n=== {var_key} ===")
    print("Atributos da variável t2m:")
    print(ds['t2m'].attrs)
    # Ver um valor específico para comparar
    val = float(ds['t2m'].isel(valid_time=0, latitude=100, longitude=100).values)
    print(f"Valor ponto [0,100,100]: {val - 273.15:.2f} °C")

In [ ]:
# ERA5 single levels tem mn2t (minimum 2m temperature) e mx2t (maximum)
# Vamos submeter os dois de uma vez

import requests

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

BASE_PAYLOAD = {
    "product_type": "monthly_averaged_reanalysis",
    "year": [str(a) for a in range(1991, 2021)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "time": "00:00",
    "area": [6, -74, -34, -28],
    "data_format": "netcdf",
}

jobs = {}

for var_key, var_name in [
    ("t2m_min", "minimum_2m_temperature_since_previous_post_processing"),
    ("t2m_max", "maximum_2m_temperature_since_previous_post_processing"),
]:
    r = requests.post(
        f"{BASE}/retrieve/v1/processes/reanalysis-era5-single-levels-monthly-means/execution",
        json={"inputs": {**BASE_PAYLOAD, "variable": var_name}},
        headers=headers)
    print(f"{var_key}: status {r.status_code}")
    jobs[var_key] = r.json().get("jobID")
    print(f"  job: {jobs[var_key]}")

print("\nJobs:", jobs)

In [ ]:
import requests, time, zipfile, os, shutil

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

DIR = "/content/drive/MyDrive/era5_brasil"

jobs = {
    't2m_min': '4491ec1d-338c-453c-b0bc-5909e8999169',
    't2m_max': '254a9e0c-aa09-4666-97c3-1da96ad142c4',
}

def wait_download(var_key, job_id, timeout=1800):
    print(f"\n⏳ {var_key}...")
    for _ in range(timeout // 15):
        r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}", headers=headers)
        status = r.json().get("status")
        print(f"  {status}")
        if status == "successful":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            url = r2.json()["asset"]["value"]["href"]
            zip_path = f"/tmp/{var_key}.zip"
            r3 = requests.get(url, stream=True)
            with open(zip_path, "wb") as f:
                for chunk in r3.iter_content(8192): f.write(chunk)
            with zipfile.ZipFile(zip_path, "r") as z:
                names = z.namelist()
                z.extractall(f"/tmp/{var_key}/")
            nc_src = f"/tmp/{var_key}/{names[0]}"
            nc_dst = os.path.join(DIR, f"brasil_{var_key}_1991_2020.nc")
            shutil.copy2(nc_src, nc_dst)
            os.remove(zip_path)
            os.remove(nc_src)
            print(f"  ✓ Salvo: {nc_dst}")
            return True
        elif status == "failed":
            r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
            print(f"  ✗ Falhou: {r2.json()}")
            return False
        time.sleep(15)
    return False

for var_key, job_id in jobs.items():
    wait_download(var_key, job_id)

# Verificar Drive
print("\nArquivos no Drive:")
for f in sorted(os.listdir(DIR)):
    size = os.path.getsize(f"{DIR}/{f}") / 1e6
    print(f"  {f} — {size:.1f} MB")

In [ ]:
import requests, zipfile, os, shutil
from google.colab import drive
drive.mount('/content/drive')

DIR = "/content/drive/MyDrive/era5_brasil"
os.makedirs(DIR, exist_ok=True)

# WorldClim v2.1 — Tmin e Tmax mensais, resolução 10 min (~18km), Brasil incluso
urls = {
    "tmin": "https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_tmin.zip",
    "tmax": "https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_tmax.zip",
}

for var, url in urls.items():
    print(f"↓ Baixando WorldClim {var}...")
    zip_path = f"/tmp/worldclim_{var}.zip"
    r = requests.get(url, stream=True)
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(8192*16):
            f.write(chunk)
    print(f"  ✓ Baixado — extraindo...")
    out_dir = os.path.join(DIR, f"worldclim_{var}")
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        print("  Arquivos:", z.namelist())
        z.extractall(out_dir)
    os.remove(zip_path)
    print(f"  ✓ Extraído em: {out_dir}")

print("\n✓ WorldClim concluído!")

In [ ]:
!pip install rasterio xarray rioxarray netCDF4 -q

import rasterio
import numpy as np
import os

DIR = "/content/drive/MyDrive/era5_brasil"

# Verificar um arquivo de cada
for var in ["tmin", "tmax"]:
    path = f"{DIR}/worldclim_{var}/wc2.1_10m_{var}_01.tif"
    with rasterio.open(path) as src:
        print(f"\nWorldClim {var} - Janeiro:")
        print(f"  CRS: {src.crs}")
        print(f"  Resolução: {src.res}")
        print(f"  Dimensões: {src.width} x {src.height}")
        print(f"  Extent: {src.bounds}")
        # Ler dados e ver range
        data = src.read(1, masked=True)
        print(f"  Valores (°C): min={float(data.min()):.1f} max={float(data.max()):.1f}")

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
import xarray as xr
import os

DIR = "/content/drive/MyDrive/era5_brasil"

# Bounding box do Brasil (mesmo do ERA5)
BBOX = {"left": -74, "bottom": -34, "right": -28, "top": 6}
MESES = [f"{m:02d}" for m in range(1, 13)]

for var in ["tmin", "tmax"]:
    print(f"\nProcessando {var}...")
    arrays = []

    for mes in MESES:
        path = f"{DIR}/worldclim_{var}/wc2.1_10m_{var}_{mes}.tif"
        with rasterio.open(path) as src:
            window = from_bounds(**BBOX, transform=src.transform)
            data = src.read(1, window=window, masked=True)
            # Substituir nodata por NaN
            data = np.where(data == src.nodata, np.nan, data.data).astype(np.float32)
            arrays.append(data)
            if mes == "01":
                # Pegar coordenadas do primeiro mês
                transform = src.window_transform(window)
                height, width = data.shape
                lons = np.array([transform.c + transform.a * (i + 0.5) for i in range(width)])
                lats = np.array([transform.f + transform.e * (j + 0.5) for j in range(height)])
                print(f"  Shape: {data.shape}, Lat: {lats[0]:.1f}→{lats[-1]:.1f}, Lon: {lons[0]:.1f}→{lons[-1]:.1f}")

    # Montar DataArray com dimensão de mês (1–12)
    arr_3d = np.stack(arrays, axis=0)  # (12, lat, lon)
    da = xr.DataArray(
        arr_3d,
        dims=["month", "lat", "lon"],
        coords={
            "month": list(range(1, 13)),
            "lat": lats,
            "lon": lons,
        },
        name=var,
        attrs={"units": "°C", "source": "WorldClim v2.1 10min"}
    )

    # Salvar como NetCDF
    out = f"{DIR}/worldclim_{var}_brasil.nc"
    da.to_netcdf(out)
    print(f"  ✓ Salvo: {out} ({os.path.getsize(out)/1e6:.1f} MB)")

print("\n✓ WorldClim processado!")

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
import xarray as xr
import os

DIR = "/content/drive/MyDrive/era5_brasil"
BBOX = {"left": -74, "bottom": -34, "right": -28, "top": 6}
MESES = [f"{m:02d}" for m in range(1, 13)]

for var in ["tmin", "tmax"]:
    print(f"\nProcessando {var}...")
    arrays = []

    for mes in MESES:
        path = f"{DIR}/worldclim_{var}/wc2.1_10m_{var}_{mes}.tif"
        with rasterio.open(path) as src:
            window = from_bounds(**BBOX, transform=src.transform)
            data = src.read(1, window=window)
            transform = src.window_transform(window)
            height, width = data.shape

            # Mascarar nodata corretamente
            nodata = src.nodata if src.nodata is not None else -3.4e+38
            data = data.astype(np.float32)
            data[data <= -9999] = np.nan       # WorldClim usa -9999 como nodata
            data[data < -200]   = np.nan       # segurança extra

            arrays.append(data)

            if mes == "01":
                lons = np.array([transform.c + transform.a * (i + 0.5) for i in range(width)])
                lats = np.array([transform.f + transform.e * (j + 0.5) for j in range(height)])
                print(f"  Shape: {data.shape}")
                print(f"  Range válido: {np.nanmin(data):.1f} a {np.nanmax(data):.1f} °C")

    arr_3d = np.stack(arrays, axis=0)
    da = xr.DataArray(
        arr_3d,
        dims=["month", "lat", "lon"],
        coords={"month": list(range(1, 13)), "lat": lats, "lon": lons},
        name=var,
        attrs={"units": "degrees_C", "source": "WorldClim v2.1 10min"}
    )

    out = f"{DIR}/worldclim_{var}_brasil.nc"
    da.to_netcdf(out)
    print(f"  ✓ Salvo: {out}")

print("\n✓ Concluído!")

In [ ]:
!pip install xarray netCDF4 rasterio numpy -q

import xarray as xr
import numpy as np
import os

DIR = "/content/drive/MyDrive/era5_brasil"

# ── Carregar dados ────────────────────────────────────────────────────────────
print("Carregando dados...")

ds_tmean  = xr.open_dataset(f"{DIR}/brasil_t2m_mean_1991_2020.nc",  engine="netcdf4")
ds_precip = xr.open_dataset(f"{DIR}/brasil_precip_1991_2020.nc",    engine="netcdf4")
ds_et     = xr.open_dataset(f"{DIR}/brasil_et_1991_2020.nc",        engine="netcdf4")
ds_tmin   = xr.open_dataset(f"{DIR}/worldclim_tmin_brasil.nc",      engine="netcdf4")
ds_tmax   = xr.open_dataset(f"{DIR}/worldclim_tmax_brasil.nc",      engine="netcdf4")

# Converter unidades
tmean  = ds_tmean['t2m']  - 273.15        # K → °C
precip = ds_precip['tp']  * 1000 * 30     # m/dia → mm/mês (aprox)
et     = ds_et['pev'].values              # já em m, vamos converter abaixo
tmin   = ds_tmin['tmin']                  # já em °C
tmax   = ds_tmax['tmax']                  # já em °C

print("✓ Dados carregados")
print(f"  Tmean shape: {tmean.shape} | coords: {list(tmean.coords)}")
print(f"  Precip shape: {precip.shape}")
print(f"  Tmin shape: {tmin.shape} | coords: {list(tmin.coords)}")

# ── Climatologia ERA5 (média 30 anos por mês) ─────────────────────────────────
print("\nCalculando climatologia mensal ERA5...")

# ERA5 usa 'valid_time' como coordenada de tempo
tmean_clim  = tmean.groupby('valid_time.month').mean('valid_time')   # (12, lat, lon)
precip_clim = precip.groupby('valid_time.month').mean('valid_time')

# ETP: pev é negativo no ERA5 (convenção de fluxo), pegar valor absoluto
et_arr = ds_et['pev']
et_mm  = np.abs(et_arr) * 1000 * 30       # m/dia → mm/mês
et_clim = et_mm.groupby('valid_time.month').mean('valid_time')

print(f"  Tmean climatologia: {tmean_clim.shape}")
print(f"  Precip climatologia: {precip_clim.shape}")
print(f"  ET climatologia: {et_clim.shape}")

# ── Variáveis derivadas ───────────────────────────────────────────────────────
print("\nCalculando variáveis derivadas...")

# 1. Déficit hídrico mensal (mm) — positivo = déficit, negativo = excedente
deficit_hidrico = et_clim - precip_clim
print(f"  Déficit hídrico: min={float(deficit_hidrico.min()):.0f} max={float(deficit_hidrico.max()):.0f} mm")

# 2. Déficit hídrico anual
deficit_anual = deficit_hidrico.sum('month')
print(f"  Déficit anual: min={float(deficit_anual.min()):.0f} max={float(deficit_anual.max()):.0f} mm/ano")

# 3. Precipitação anual
precip_anual = precip_clim.sum('month')
print(f"  Precip anual: min={float(precip_anual.min()):.0f} max={float(precip_anual.max()):.0f} mm/ano")

# 4. Temperatura média anual
tmean_anual = tmean_clim.mean('month')
print(f"  Tmean anual: min={float(tmean_anual.min()):.1f} max={float(tmean_anual.max()):.1f} °C")

# 5. Chilling hours mensais (estimativa — modelo de Utah simplificado)
# Horas com T entre 0 e 7.2°C, estimadas a partir de Tmin e Tmean diários
# Usando WorldClim Tmin como proxy: se Tmin_mes < 7.2°C, há potencial de chilling
# Fórmula: CH_mes = dias_mes × 24 × P(0 < T < 7.2°C)
# P estimada via distribuição normal com média=Tmean e desvio=(Tmean-Tmin)/2

dias_mes = xr.DataArray(
    [31,28,31,30,31,30,31,31,30,31,30,31],
    dims=['month'], coords={'month': range(1,13)}
)

# Interpolar Tmin WorldClim para o grid do ERA5 (necessário para operar juntos)
tmin_interp = tmin.interp(
    lat=tmean_clim.latitude,
    lon=tmean_clim.longitude,
    method='linear'
)
tmin_interp = tmin_interp.rename({'month': 'month'})

# Desvio padrão estimado da temperatura diária
sigma = (tmean_clim - tmin_interp) / 2
sigma = sigma.rename({'latitude': 'latitude', 'longitude': 'longitude'})

# Probabilidade de T estar entre 0 e 7.2°C usando CDF normal
from scipy import stats as sp_stats
import warnings
warnings.filterwarnings('ignore')

# Calcular para cada mês
ch_list = []
for m in range(1, 13):
    t_m   = float(tmean_clim.sel(month=m).mean())   # escalar para teste
    s_m   = float(sigma.sel(month=m).mean())
    if s_m <= 0: s_m = 1.0
    p_ch  = sp_stats.norm.cdf(7.2, t_m, s_m) - sp_stats.norm.cdf(0, t_m, s_m)
    ch_m  = dias_mes.sel(month=m).values * 24 * p_ch
    ch_list.append(float(ch_m))

ch_anual_media = sum(ch_list)
print(f"\n  Chilling hours anuais (média Brasil): {ch_anual_media:.0f} h")
print(f"  Por mês: { {m+1: round(ch_list[m]) for m in range(12)} }")

# ── Salvar climatologia final ─────────────────────────────────────────────────
print("\nSalvando climatologia...")

ds_clim = xr.Dataset({
    'tmean':   tmean_clim.rename({'latitude':'lat','longitude':'lon'}),
    'precip':  precip_clim.rename({'latitude':'lat','longitude':'lon'}),
    'et':      et_clim.rename({'latitude':'lat','longitude':'lon'}),
    'deficit': deficit_hidrico.rename({'latitude':'lat','longitude':'lon'}),
})

out = f"{DIR}/climatologia_brasil_1991_2020.nc"
ds_clim.to_netcdf(out)
print(f"✓ Salvo: {out} ({os.path.getsize(out)/1e6:.1f} MB)")

# Salvar Tmin e Tmax WorldClim separadamente (já no formato correto)
print("\n✓ Pipeline de climatologia concluído!")
print("\nResumo das variáveis disponíveis para o Mahalanobis:")
print("  tmean  — temperatura média mensal (°C)")
print("  tmin   — temperatura mínima mensal (°C) [WorldClim]")
print("  tmax   — temperatura máxima mensal (°C) [WorldClim]")
print("  precip — precipitação mensal (mm)")
print("  et     — evapotranspiração mensal (mm)")
print("  deficit— déficit hídrico mensal (mm)")

In [ ]:
!pip install xarray netCDF4 rasterio scipy -q

import xarray as xr
import numpy as np
from scipy import stats as sp_stats
import os

DIR = "/content/drive/MyDrive/era5_brasil"

# Carregar climatologia ERA5
ds_clim = xr.open_dataset(f"{DIR}/climatologia_brasil_1991_2020.nc", engine="netcdf4")

# Carregar WorldClim Tmin e Tmax
ds_tmin = xr.open_dataset(f"{DIR}/worldclim_tmin_brasil.nc", engine="netcdf4")
ds_tmax = xr.open_dataset(f"{DIR}/worldclim_tmax_brasil.nc", engine="netcdf4")

# Interpolar WorldClim para o grid ERA5
print("Interpolando WorldClim para grid ERA5...")
tmin_interp = ds_tmin['tmin'].interp(
    lat=ds_clim['lat'], lon=ds_clim['lon'], method='linear'
)
tmax_interp = ds_tmax['tmax'].interp(
    lat=ds_clim['lat'], lon=ds_clim['lon'], method='linear'
)
print(f"  Tmin interpolado: {tmin_interp.shape}")
print(f"  Tmax interpolado: {tmax_interp.shape}")
print(f"  Tmin range: {float(np.nanmin(tmin_interp)):.1f} a {float(np.nanmax(tmin_interp)):.1f} °C")
print(f"  Tmax range: {float(np.nanmin(tmax_interp)):.1f} a {float(np.nanmax(tmax_interp)):.1f} °C")

# Calcular chilling hours pixel a pixel
print("\nCalculando chilling hours pixel a pixel...")
dias_mes = [31,28,31,30,31,30,31,31,30,31,30,31]

ch_list = []
for i, m in enumerate(range(1, 13)):
    tmean_m = ds_clim['tmean'].sel(month=m).values   # (lat, lon)
    tmin_m  = tmin_interp.sel(month=m).values         # (lat, lon)

    sigma = (tmean_m - tmin_m) / 2
    sigma = np.where(sigma <= 0, 1.0, sigma)

    # P(0 < T < 7.2°C) pixel a pixel
    p_ch = (sp_stats.norm.cdf(7.2, tmean_m, sigma) -
            sp_stats.norm.cdf(0.0,  tmean_m, sigma))
    ch_m = dias_mes[i] * 24 * p_ch
    ch_list.append(ch_m)

ch_array = np.stack(ch_list, axis=0)  # (12, lat, lon)
ch_anual = ch_array.sum(axis=0)       # (lat, lon)

print(f"  Chilling hours anuais: min={np.nanmin(ch_anual):.0f} max={np.nanmax(ch_anual):.0f} h")
print(f"  Sul do Brasil (média): {np.nanmean(ch_anual[300:, :]):.0f} h")

# Montar dataset final completo
print("\nSalvando dataset completo...")
ds_final = xr.Dataset({
    'tmean':   ds_clim['tmean'],
    'tmin':    tmin_interp,
    'tmax':    tmax_interp,
    'precip':  ds_clim['precip'],
    'et':      ds_clim['et'],
    'deficit': ds_clim['deficit'],
    'chilling_hours': xr.DataArray(
        ch_array,
        dims=['month','lat','lon'],
        coords={'month': range(1,13),
                'lat': ds_clim['lat'],
                'lon': ds_clim['lon']},
        attrs={'units': 'hours', 'description': 'Estimated chilling hours (0-7.2°C)'}
    ),
    'chilling_hours_anual': xr.DataArray(
        ch_anual,
        dims=['lat','lon'],
        coords={'lat': ds_clim['lat'], 'lon': ds_clim['lon']},
        attrs={'units': 'hours/year'}
    ),
})

out = f"{DIR}/climatologia_completa_brasil.nc"
ds_final.to_netcdf(out)
print(f"✓ Salvo: {out} ({os.path.getsize(out)/1e6:.1f} MB)")

print("\n✓ Dataset completo com 7 variáveis pronto para o Mahalanobis!")
print("  tmean, tmin, tmax, precip, et, deficit, chilling_hours")

In [ ]:
!pip install xarray netCDF4 numpy scipy matplotlib cartopy -q

import xarray as xr
import numpy as np
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

DIR = "/content/drive/MyDrive/era5_brasil"

# ── Carregar climatologia ─────────────────────────────────────────────────────
ds = xr.open_dataset(f"{DIR}/climatologia_completa_brasil.nc", engine="netcdf4")
print("Dataset carregado:", list(ds.data_vars))

# ── Requerimentos ecofisiológicos das 6 culturas ──────────────────────────────
# Formato: [tmean_anual, tmin_mes_frio, tmax_mes_quente, precip_anual, deficit_anual, chilling_anual]
# Baseado em literatura (Allen 1998, Mehlenbacher 1991, Ferguson 2002, etc.)

CULTURAS = {
    "Lúpulo": {
        "tmean_anual":  (14, 22),    # °C ótimo
        "tmin_min":     (-20, 5),    # Tmin mês mais frio
        "tmax_max":     (20, 30),    # Tmax mês mais quente
        "precip_anual": (310, 1370), # mm/ano
        "chilling":     (500, 2000), # horas/ano
    },
    "Pistache": {
        "tmean_anual":  (18, 35),
        "tmin_min":     (-10, 5),
        "tmax_max":     (30, 45),
        "precip_anual": (250, 600),
        "chilling":     (700, 1500),
    },
    "Avelã": {
        "tmean_anual":  (10, 20),
        "tmin_min":     (-15, 5),
        "tmax_max":     (20, 32),
        "precip_anual": (600, 1200),
        "chilling":     (700, 1800),
    },
    "Amêndoa": {
        "tmean_anual":  (15, 32),
        "tmin_min":     (-4, 8),
        "tmax_max":     (25, 38),
        "precip_anual": (300, 600),
        "chilling":     (250, 600),
    },
    "Açafrão": {
        "tmean_anual":  (10, 20),
        "tmin_min":     (-10, 5),
        "tmax_max":     (20, 30),
        "precip_anual": (400, 700),
        "chilling":     (600, 1200),
    },
    "Trufas": {
        "tmean_anual":  (10, 20),
        "tmin_min":     (-5, 5),
        "tmax_max":     (20, 32),
        "precip_anual": (600, 900),
        "chilling":     (600, 1000),
    },
}

# ── Preparar variáveis climáticas por pixel ───────────────────────────────────
print("\nPreparando variáveis climáticas...")

tmean_anual  = ds['tmean'].mean('month').values              # (lat, lon)
tmin_min     = ds['tmin'].min('month').values                # Tmin do mês mais frio
tmax_max     = ds['tmax'].max('month').values                # Tmax do mês mais quente
precip_anual = ds['precip'].sum('month').values              # mm/ano
ch_anual     = ds['chilling_hours_anual'].values             # h/ano

lat = ds['lat'].values
lon = ds['lon'].values
nlat, nlon = len(lat), len(lon)

print(f"  Grid: {nlat} x {nlon} = {nlat*nlon:,} pixels")

# ── Função de aptidão por cultura ─────────────────────────────────────────────
def calc_aptidao(req, tmean_a, tmin_m, tmax_m, precip_a, ch_a):
    """
    Calcula índice de aptidão (0-1) baseado em distância normalizada
    para cada variável climática. Produto das aptidões individuais.
    """
    def score_var(val, vmin, vmax):
        """Score 0-1: 1 se dentro do range ótimo, decresce para fora"""
        meio = (vmin + vmax) / 2
        amp  = (vmax - vmin) / 2
        dist = np.abs(val - meio) / amp
        return np.clip(1 - dist, 0, 1)

    s1 = score_var(tmean_a,  *req["tmean_anual"])
    s2 = score_var(tmin_m,   *req["tmin_min"])
    s3 = score_var(tmax_m,   *req["tmax_max"])
    s4 = score_var(precip_a, *req["precip_anual"])
    s5 = score_var(ch_a,     *req["chilling"])

    # Aptidão final = média ponderada (todas com peso igual por ora)
    aptidao = (s1 + s2 + s3 + s4 + s5) / 5
    return aptidao

# ── Calcular aptidão para cada cultura ───────────────────────────────────────
print("\nCalculando aptidão por cultura...")
resultados = {}

for cultura, req in CULTURAS.items():
    apt = calc_aptidao(req, tmean_anual, tmin_min, tmax_max, precip_anual, ch_anual)
    resultados[cultura] = apt
    area_apta = np.sum(apt > 0.5) / (nlat * nlon) * 100
    area_otima = np.sum(apt > 0.7) / (nlat * nlon) * 100
    print(f"  {cultura}: área apta (>0.5): {area_apta:.1f}% | ótima (>0.7): {area_otima:.1f}%")

# ── Salvar resultados ─────────────────────────────────────────────────────────
print("\nSalvando mapas de aptidão...")
ds_apt = xr.Dataset(
    {cultura: xr.DataArray(apt, dims=['lat','lon'],
                           coords={'lat': lat, 'lon': lon},
                           attrs={'units': '0-1', 'cultura': cultura})
     for cultura, apt in resultados.items()}
)
out = f"{DIR}/aptidao_atual_brasil.nc"
ds_apt.to_netcdf(out)
print(f"✓ Salvo: {out}")

# ── Plotar mapas ──────────────────────────────────────────────────────────────
print("\nGerando mapas...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

cmap = plt.cm.RdYlGn
norm = mcolors.Normalize(vmin=0, vmax=1)

for i, (cultura, apt) in enumerate(resultados.items()):
    ax = axes[i]
    im = ax.pcolormesh(lon, lat, apt, cmap=cmap, norm=norm)
    ax.set_title(cultura, fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(im, ax=ax, label='Aptidão (0-1)')

plt.suptitle('Aptidão Climática Atual — Brasil (1991–2020)', fontsize=15, fontweight='bold')
plt.tight_layout()

out_fig = f"{DIR}/mapas_aptidao_atual.png"
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Figura salva: {out_fig}")

In [ ]:
import xarray as xr
import numpy as np

DIR = "/content/drive/MyDrive/era5_brasil"
ds = xr.open_dataset(f"{DIR}/climatologia_completa_brasil.nc", engine="netcdf4")

# Diagnóstico por variável para entender o gargalo
tmean_anual  = ds['tmean'].mean('month').values
tmin_min     = ds['tmin'].min('month').values
tmax_max     = ds['tmax'].max('month').values
precip_anual = ds['precip'].sum('month').values
ch_anual     = ds['chilling_hours_anual'].values

lat = ds['lat'].values
lon = ds['lon'].values

# Focar no Sul do Brasil (região mais promissora)
# Lat -25 a -34 = índices aproximados 310 a 400
i_sul = slice(310, 401)

print("=== DIAGNÓSTICO — SUL DO BRASIL ===")
print(f"Tmean anual:  {np.nanmean(tmean_anual[i_sul]):.1f} °C  (range: {np.nanmin(tmean_anual[i_sul]):.1f} a {np.nanmax(tmean_anual[i_sul]):.1f})")
print(f"Tmin mês frio: {np.nanmean(tmin_min[i_sul]):.1f} °C  (range: {np.nanmin(tmin_min[i_sul]):.1f} a {np.nanmax(tmin_min[i_sul]):.1f})")
print(f"Tmax mês qte: {np.nanmean(tmax_max[i_sul]):.1f} °C  (range: {np.nanmin(tmax_max[i_sul]):.1f} a {np.nanmax(tmax_max[i_sul]):.1f})")
print(f"Precip anual: {np.nanmean(precip_anual[i_sul]):.0f} mm  (range: {np.nanmin(precip_anual[i_sul]):.0f} a {np.nanmax(precip_anual[i_sul]):.0f})")
print(f"Chilling hrs: {np.nanmean(ch_anual[i_sul]):.0f} h  (range: {np.nanmin(ch_anual[i_sul]):.0f} a {np.nanmax(ch_anual[i_sul]):.0f})")

print("\n=== REQUERIMENTOS vs REALIDADE — LÚPULO ===")
req_lupulo = {"tmean": (14,22), "tmin": (-20,5), "tmax": (20,30), "precip": (310,1370), "chilling": (500,2000)}
print(f"Tmean Sul:   {np.nanmean(tmean_anual[i_sul]):.1f}°C  | req: {req_lupulo['tmean']}  → {'✓' if req_lupulo['tmean'][0] < np.nanmean(tmean_anual[i_sul]) < req_lupulo['tmean'][1] else '✗'}")
print(f"Tmin Sul:    {np.nanmean(tmin_min[i_sul]):.1f}°C   | req: {req_lupulo['tmin']}  → {'✓' if req_lupulo['tmin'][0] < np.nanmean(tmin_min[i_sul]) < req_lupulo['tmin'][1] else '✗'}")
print(f"Tmax Sul:    {np.nanmean(tmax_max[i_sul]):.1f}°C   | req: {req_lupulo['tmax']}  → {'✓' if req_lupulo['tmax'][0] < np.nanmean(tmax_max[i_sul]) < req_lupulo['tmax'][1] else '✗'}")
print(f"Precip Sul:  {np.nanmean(precip_anual[i_sul]):.0f}mm  | req: {req_lupulo['precip']}  → {'✓' if req_lupulo['precip'][0] < np.nanmean(precip_anual[i_sul]) < req_lupulo['precip'][1] else '✗'}")
print(f"Chilling Sul:{np.nanmean(ch_anual[i_sul]):.0f}h   | req: {req_lupulo['chilling']}  → {'✓' if req_lupulo['chilling'][0] < np.nanmean(ch_anual[i_sul]) < req_lupulo['chilling'][1] else '✗'}")

print("\n=== DIAGNÓSTICO — AMÊNDOA (mais ampla) ===")
req_amendoa = {"tmean": (15,32), "tmin": (-4,8), "tmax": (25,38), "precip": (300,600), "chilling": (250,600)}
print(f"Tmean Sul:   {np.nanmean(tmean_anual[i_sul]):.1f}°C  | req: {req_amendoa['tmean']}  → {'✓' if req_amendoa['tmean'][0] < np.nanmean(tmean_anual[i_sul]) < req_amendoa['tmean'][1] else '✗'}")
print(f"Tmin Sul:    {np.nanmean(tmin_min[i_sul]):.1f}°C   | req: {req_amendoa['tmin']}  → {'✓' if req_amendoa['tmin'][0] < np.nanmean(tmin_min[i_sul]) < req_amendoa['tmin'][1] else '✗'}")
print(f"Tmax Sul:    {np.nanmean(tmax_max[i_sul]):.1f}°C   | req: {req_amendoa['tmax']}  → {'✓' if req_amendoa['tmax'][0] < np.nanmean(tmax_max[i_sul]) < req_amendoa['tmax'][1] else '✗'}")
print(f"Precip Sul:  {np.nanmean(precip_anual[i_sul]):.0f}mm  | req: {req_amendoa['precip']}  → {'✓' if req_amendoa['precip'][0] < np.nanmean(precip_anual[i_sul]) < req_amendoa['precip'][1] else '✗'}")
print(f"Chilling Sul:{np.nanmean(ch_anual[i_sul]):.0f}h   | req: {req_amendoa['chilling']}  → {'✓' if req_amendoa['chilling'][0] < np.nanmean(ch_anual[i_sul]) < req_amendoa['chilling'][1] else '✗'}")

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

DIR = "/content/drive/MyDrive/era5_brasil"
ds = xr.open_dataset(f"{DIR}/climatologia_completa_brasil.nc", engine="netcdf4")

tmean_anual  = ds['tmean'].mean('month').values
tmin_min     = ds['tmin'].min('month').values
tmax_max     = ds['tmax'].max('month').values
precip_anual = ds['precip'].sum('month').values
ch_anual     = ds['chilling_hours_anual'].values
lat = ds['lat'].values
lon = ds['lon'].values

CULTURAS = {
    "Lúpulo": {
        "tmean_anual":  (12, 14, 22, 25),
        "tmin_min":     (-20, -15, 7, 10),
        "tmax_max":     (18, 20, 32, 36),
        "precip_anual": (200, 310, 1370, 1600),
        "chilling":     (300, 500, 2000, 2500),
    },
    "Pistache": {
        "tmean_anual":  (15, 18, 35, 38),
        "tmin_min":     (-12, -10, 7, 10),
        "tmax_max":     (25, 30, 45, 48),
        "precip_anual": (150, 250, 600, 800),
        "chilling":     (500, 700, 1500, 1800),
    },
    "Avelã": {
        "tmean_anual":  (8, 10, 20, 23),
        "tmin_min":     (-18, -15, 7, 10),
        "tmax_max":     (18, 20, 34, 37),
        "precip_anual": (400, 600, 1200, 1500),
        "chilling":     (500, 700, 1800, 2200),
    },
    "Amêndoa": {
        "tmean_anual":  (12, 15, 32, 35),
        "tmin_min":     (-6, -4, 10, 13),
        "tmax_max":     (22, 25, 38, 41),
        "precip_anual": (200, 300, 600, 800),
        "chilling":     (150, 250, 600, 800),
    },
    "Açafrão": {
        "tmean_anual":  (8, 10, 20, 23),
        "tmin_min":     (-12, -10, 7, 10),
        "tmax_max":     (18, 20, 32, 35),
        "precip_anual": (300, 400, 700, 900),
        "chilling":     (400, 600, 1200, 1500),
    },
    "Trufas": {
        "tmean_anual":  (8, 10, 20, 23),
        "tmin_min":     (-7, -5, 7, 10),
        "tmax_max":     (18, 20, 32, 35),
        "precip_anual": (400, 600, 900, 1100),
        "chilling":     (400, 600, 1000, 1300),
    },
}

def score_trapezio(val, a, b, c, d):
    """
    Função trapezoidal:
    0 fora de [a,d], sobe linearmente de a a b,
    1 entre b e c, desce linearmente de c a d.
    """
    return np.where(
        val <= a, 0.0,
        np.where(val <= b, (val - a) / (b - a),
        np.where(val <= c, 1.0,
        np.where(val <= d, (d - val) / (d - c),
        0.0)))
    ).astype(float)

def calc_aptidao(req, tmean_a, tmin_m, tmax_m, precip_a, ch_a):
    s1 = score_trapezio(tmean_a,  *req["tmean_anual"])
    s2 = score_trapezio(tmin_m,   *req["tmin_min"])
    s3 = score_trapezio(tmax_m,   *req["tmax_max"])
    s4 = score_trapezio(precip_a, *req["precip_anual"])
    s5 = score_trapezio(ch_a,     *req["chilling"])
    return (s1 + s2 + s3 + s4 + s5) / 5

# Calcular e diagnosticar
print("Aptidão por cultura — Sul do Brasil:")
i_sul = slice(310, 401)
resultados = {}

for cultura, req in CULTURAS.items():
    apt = calc_aptidao(req, tmean_anual, tmin_min, tmax_max, precip_anual, ch_anual)
    resultados[cultura] = apt
    apt_sul = apt[i_sul]
    print(f"  {cultura}: Sul média={np.nanmean(apt_sul):.2f} | apta(>0.5)={np.sum(apt_sul>0.5)/apt_sul.size*100:.1f}% | ótima(>0.7)={np.sum(apt_sul>0.7)/apt_sul.size*100:.1f}%")

# Salvar
ds_apt = xr.Dataset({
    cultura: xr.DataArray(apt, dims=['lat','lon'],
                          coords={'lat': lat, 'lon': lon})
    for cultura, apt in resultados.items()
})
ds_apt.to_netcdf(f"{DIR}/aptidao_atual_brasil.nc")

# Plotar
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
cmap = plt.cm.RdYlGn
norm = mcolors.Normalize(vmin=0, vmax=1)

for i, (cultura, apt) in enumerate(resultados.items()):
    ax = axes[i]
    im = ax.pcolormesh(lon, lat, apt, cmap=cmap, norm=norm)
    ax.set_title(cultura, fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    # Linha de referência no Sul
    ax.axhline(-25, color='blue', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axhline(-34, color='blue', linewidth=0.8, linestyle='--', alpha=0.5)
    plt.colorbar(im, ax=ax, label='Aptidão (0-1)')

plt.suptitle('Aptidão Climática Atual — Brasil (1991–2020)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{DIR}/mapas_aptidao_v2.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Mapas salvos!")

In [ ]:
import xarray as xr
import numpy as np

DIR = "/content/drive/MyDrive/era5_brasil"

# Carregar aptidão
ds_apt = xr.open_dataset(f"{DIR}/aptidao_atual_brasil.nc", engine="netcdf4")
lat = ds_apt['lat'].values
lon = ds_apt['lon'].values

# Carregar máscara do Brasil via shapefile
import requests, zipfile, os

# Baixar shapefile do Brasil (IBGE)
if not os.path.exists("/tmp/brasil.shp"):
    url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/Brasil/BR/BR_Municipios_2022.zip"
    print("Baixando shapefile IBGE...")
    r = requests.get(url, stream=True)
    with open("/tmp/brasil_mun.zip", "wb") as f:
        for chunk in r.iter_content(8192*16): f.write(chunk)
    with zipfile.ZipFile("/tmp/brasil_mun.zip") as z:
        z.extractall("/tmp/brasil_shp/")
    print("✓ Extraído")

import glob
shp = glob.glob("/tmp/brasil_shp/*.shp")[0]
print("Shapefile:", shp)

# Criar máscara raster do Brasil
!pip install geopandas rasterio -q
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from shapely.geometry import mapping

br = gpd.read_file(shp).dissolve()
transform = from_bounds(lon.min(), lat.min(), lon.max(), lat.max(), len(lon), len(lat))
mascara = rasterize(
    [(mapping(geom), 1) for geom in br.geometry],
    out_shape=(len(lat), len(lon)),
    transform=transform,
    fill=0, dtype='uint8'
)
# Corrigir orientação (lat decrescente)
mascara = mascara[::-1]
print(f"Máscara: {mascara.sum():,} pixels brasileiros de {mascara.size:,} total ({mascara.sum()/mascara.size*100:.1f}%)")

# Estatísticas só dentro do Brasil
print("\n=== APTIDÃO — TERRITÓRIO BRASILEIRO ===")
culturas = list(ds_apt.data_vars)
for cultura in culturas:
    apt = ds_apt[cultura].values
    apt_br = apt[mascara == 1]
    apt_sul_br = apt[(mascara == 1) & (lat[:, None] < -20).repeat(len(lon), axis=1)]
    print(f"\n{cultura}:")
    print(f"  Brasil todo: apta(>0.5)={np.sum(apt_br>0.5)/len(apt_br)*100:.1f}% | ótima(>0.7)={np.sum(apt_br>0.7)/len(apt_br)*100:.1f}%")
    print(f"  Sul (<-20°): apta(>0.5)={np.sum(apt_sul_br>0.5)/len(apt_sul_br)*100:.1f}% | ótima(>0.7)={np.sum(apt_sul_br>0.7)/len(apt_sul_br)*100:.1f}%")
    print(f"  Máximo:      {np.nanmax(apt_br):.2f} | Média apta: {np.nanmean(apt_br[apt_br>0.5]):.2f}" if np.any(apt_br>0.5) else "  Sem área apta")

In [ ]:
import numpy as np

# Área de um pixel a 0.1° na latitude média do Sul do Brasil (~27°S)
# 1° latitude ≈ 111 km, 1° longitude ≈ 111 × cos(27°) ≈ 99 km
# pixel = 0.1° × 0.1° ≈ 11.1 km × 9.9 km ≈ 109.9 km² ≈ 10,990 ha

lat_media_sul = -27
area_pixel_ha = (111 * 0.1) * (111 * np.cos(np.radians(abs(lat_media_sul))) * 0.1) * 100
print(f"Área por pixel: {area_pixel_ha:.0f} ha")

# Pixels aptos no Sul por cultura (>0.5)
pixels_sul = 71368 * 0.386  # pixels brasileiros no Sul (abaixo de -20°)
print(f"\nPixels Sul do Brasil: {pixels_sul:.0f}")

culturas_sul = {
    "Amêndoa":  0.550,
    "Lúpulo":   0.410,
    "Avelã":    0.323,
    "Trufas":   0.250,
    "Açafrão":  0.239,
    "Pistache": 0.157,
}

print(f"\n{'Cultura':<12} {'Área apta (Mha)':>15} {'Área ótima (Mha)':>16}")
print("-" * 45)
for cultura, frac in culturas_sul.items():
    area_mha = pixels_sul * frac * area_pixel_ha / 1e6
    print(f"{cultura:<12} {area_mha:>15.1f}")

In [ ]:
import requests

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

# CMIP6 monthly — temperatura e precipitação
# Dataset: projections-cmip6

jobs = {}

for var_key, var_name in [
    ("tas",  "near_surface_air_temperature"),      # Tmean
    ("tasmin", "daily_minimum_near_surface_air_temperature"),  # Tmin
    ("tasmax", "daily_maximum_near_surface_air_temperature"),  # Tmax
    ("pr",   "precipitation"),                      # Precip
]:
    for periodo, anos in [
        ("2041_2060", list(range(2041, 2061))),
        ("2071_2090", list(range(2071, 2091))),
    ]:
        for ssp in ["ssp245", "ssp585"]:
            payload = {
                "model": "mpi_esm1_2_hr",      # modelo único para teste
                "experiment": ssp,
                "variable": var_name,
                "temporal_resolution": "monthly",
                "year": [str(a) for a in anos],
                "month": [f"{m:02d}" for m in range(1, 13)],
                "area": [6, -74, -34, -28],
                "data_format": "netcdf",
            }
            r = requests.post(
                f"{BASE}/retrieve/v1/processes/projections-cmip6/execution",
                json={"inputs": payload}, headers=headers)
            job_id = r.json().get("jobID")
            key = f"{var_key}_{ssp}_{periodo}"
            jobs[key] = job_id
            status = r.status_code
            print(f"  {key}: {status} → {job_id}")

print(f"\nTotal jobs submetidos: {len(jobs)}")

In [ ]:
r = requests.get(
    f"{BASE}/retrieve/v1/processes/projections-cmip6",
    headers=headers)
print("Status:", r.status_code)
import json
info = r.json()
print("Campos disponíveis:", list(info.get("inputs", {}).keys()))

In [ ]:
# Ver estrutura completa
import json
print(json.dumps(inputs["model"], indent=2)[:2000])
print()
print(json.dumps(inputs["variable"], indent=2)[:2000])
print()
print(json.dumps(inputs["experiment"], indent=2)[:500])

In [ ]:
import requests

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

# Verificar modelos disponíveis primeiro
r = requests.get(f"{BASE}/retrieve/v1/processes/projections-cmip6", headers=headers)
inputs = r.json()["inputs"]
models = inputs["model"]["schema"]["enum"]
print("Primeiros 15 modelos:")
for m in models[:15]:
    print(f"  {m}")

In [ ]:
# Jobs já confirmados
jobs = {
    "tas_ssp245_2041_2060":    "e9b56d56-2954-4621-b015-6a0a5fa67e0e",
    "tas_ssp585_2041_2060":    "709d4a49-7823-463e-825f-822d0cd55ed8",
    "tas_ssp245_2071_2090":    "d3fdd298-681a-450b-b204-f8014ad79ca9",
    "tas_ssp585_2071_2090":    "ef594d6a-8ad5-47b3-968e-8d68a2ca6e4c",
    "tasmin_ssp245_2041_2060": "93468a2f-fbb1-4aec-a2cf-6ea7d96675c5",
    "tasmin_ssp585_2041_2060": "2f18de6d-8399-4738-b55c-982e99ab283c",
    "tasmin_ssp245_2071_2090": "40531a95-0a16-45d6-be60-1d2fd22cb33f",
    "tasmin_ssp585_2071_2090": "38c891c3-c696-4d5f-93b7-bb89179c8b63",
    "tasmax_ssp245_2041_2060": "6c22116b-622f-483b-a41e-bb72912c5af7",
}

# Resubmeter os 7 restantes com pausa
import time

faltando = {
    "tasmax_ssp585_2041_2060": ("daily_maximum_near_surface_air_temperature", "ssp5_8_5", list(range(2041,2061))),
    "tasmax_ssp245_2071_2090": ("daily_maximum_near_surface_air_temperature", "ssp2_4_5", list(range(2071,2091))),
    "tasmax_ssp585_2071_2090": ("daily_maximum_near_surface_air_temperature", "ssp5_8_5", list(range(2071,2091))),
    "pr_ssp245_2041_2060":     ("precipitation", "ssp2_4_5", list(range(2041,2061))),
    "pr_ssp585_2041_2060":     ("precipitation", "ssp5_8_5", list(range(2041,2061))),
    "pr_ssp245_2071_2090":     ("precipitation", "ssp2_4_5", list(range(2071,2091))),
    "pr_ssp585_2071_2090":     ("precipitation", "ssp5_8_5", list(range(2071,2091))),
}

for key, (var_name, ssp, anos) in faltando.items():
    time.sleep(3)  # pausa para evitar rate limit
    payload = {
        "model": "miroc6",
        "experiment": ssp,
        "variable": var_name,
        "temporal_resolution": "monthly",
        "year": [str(a) for a in anos],
        "month": [f"{m:02d}" for m in range(1, 13)],
        "area": [6, -74, -34, -28],
        "data_format": "netcdf",
    }
    r = requests.post(
        f"{BASE}/retrieve/v1/processes/projections-cmip6/execution",
        json={"inputs": payload}, headers=headers)
    if r.status_code == 201:
        job_id = r.json().get("jobID")
        jobs[key] = job_id
        print(f"  ✓ {key}: {job_id}")
    else:
        print(f"  ✗ {key}: {r.status_code} — {r.text[:100]}")

print(f"\nTotal jobs: {len(jobs)}/16")
print(jobs)

In [ ]:
import requests, os

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

# Testar com um job
job_id = "e9b56d56-2954-4621-b015-6a0a5fa67e0e"  # tas_ssp245_2041_2060

r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
print("Status results:", r.status_code)
import json
result = r.json()
print(json.dumps(result, indent=2)[:1000])

In [ ]:
import requests, zipfile, os, shutil
from google.colab import drive
drive.mount('/content/drive')

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

DIR = "/content/drive/MyDrive/era5_brasil/cmip6"

jobs = {
    "tas_ssp245_2041_2060":    "e9b56d56-2954-4621-b015-6a0a5fa67e0e",
    "tas_ssp585_2041_2060":    "709d4a49-7823-463e-825f-822d0cd55ed8",
    "tas_ssp245_2071_2090":    "d3fdd298-681a-450b-b204-f8014ad79ca9",
    "tas_ssp585_2071_2090":    "ef594d6a-8ad5-47b3-968e-8d68a2ca6e4c",
    "tasmin_ssp245_2041_2060": "93468a2f-fbb1-4aec-a2cf-6ea7d96675c5",
    "tasmin_ssp585_2041_2060": "2f18de6d-8399-4738-b55c-982e99ab283c",
    "tasmin_ssp245_2071_2090": "40531a95-0a16-45d6-be60-1d2fd22cb33f",
    "tasmin_ssp585_2071_2090": "38c891c3-c696-4d5f-93b7-bb89179c8b63",
    "tasmax_ssp245_2041_2060": "6c22116b-622f-483b-a41e-bb72912c5af7",
    "tasmax_ssp585_2041_2060": "1118583d-0135-4ea1-ab13-f33634db139a",
    "tasmax_ssp245_2071_2090": "cac50c9e-8095-4207-87c1-9f835a61a8fd",
    "tasmax_ssp585_2071_2090": "d4b7a430-252a-44b7-ac04-9eb273410f03",
    "pr_ssp245_2041_2060":     "14ec469e-4664-473d-90f8-86f92edcb27d",
    "pr_ssp585_2041_2060":     "acc33dcc-f62f-4345-8c37-5b19debae634",
    "pr_ssp245_2071_2090":     "1fdc5aef-7cde-433e-95e3-1d0f6a3c9080",
    "pr_ssp585_2071_2090":     "547bcccf-6397-4901-b118-a459513fc268",
}

def download_job(key, job_id):
    # Pegar URL do resultado
    r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
    result = r.json()
    url  = result["asset"]["value"]["href"]
    size = result["asset"]["value"].get("file:size", 0)
    print(f"  URL obtida | tamanho esperado: {size/1e6:.1f} MB")

    # Baixar com stream e verificar tamanho
    zip_path = f"/tmp/{key}.zip"
    r2 = requests.get(url, stream=True, timeout=300)
    downloaded = 0
    with open(zip_path, "wb") as f:
        for chunk in r2.iter_content(chunk_size=65536):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
    print(f"  Baixado: {downloaded/1e6:.1f} MB")

    if downloaded < 1000:
        print(f"  ✗ Arquivo muito pequeno — pulando")
        return False

    # Extrair ZIP
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        print(f"  Arquivos no ZIP: {names}")
        z.extractall(f"/tmp/{key}/")

    # Copiar NetCDF para Drive
    nc_files = [f for f in os.listdir(f"/tmp/{key}/") if f.endswith(".nc")]
    if not nc_files:
        print(f"  ✗ Nenhum .nc encontrado")
        return False

    nc_src = f"/tmp/{key}/{nc_files[0]}"
    nc_dst = f"{DIR}/{key}.nc"
    shutil.copy2(nc_src, nc_dst)
    os.remove(zip_path)
    shutil.rmtree(f"/tmp/{key}/")
    print(f"  ✓ Salvo: {nc_dst} ({os.path.getsize(nc_dst)/1e6:.1f} MB)")
    return True

for key, job_id in jobs.items():
    print(f"\n↓ {key}")
    download_job(key, job_id)

print("\nArquivos finais:")
for f in sorted(os.listdir(DIR)):
    size = os.path.getsize(f"{DIR}/{f}")
    if size > 1000:
        print(f"  ✓ {f} — {size/1e6:.1f} MB")
    else:
        print(f"  ✗ {f} — vazio")

In [ ]:
!pip install netCDF4 xarray numpy -q

import xarray as xr
import numpy as np

DIR = "/content/drive/MyDrive/era5_brasil/cmip6"

# Verificar um arquivo
ds = xr.open_dataset(f"{DIR}/tas_ssp245_2041_2060.nc", engine="netcdf4")
print("=== tas_ssp245_2041_2060 ===")
print(ds)
print("\nVariáveis:", list(ds.data_vars))
print("Coords:", list(ds.coords))
print("Tempo - primeiro:", ds.time.values[0])
print("Tempo - último:", ds.time.values[-1])
print("Total meses:", len(ds.time))

# Ver range de temperatura
var = list(ds.data_vars)[0]
print(f"\n{var}: min={float(ds[var].min())-273.15:.1f} max={float(ds[var].max())-273.15:.1f} °C")
print(f"Lat: {float(ds.lat.min()):.1f} a {float(ds.lat.max()):.1f}")
print(f"Lon: {float(ds.lon.min()):.1f} a {float(ds.lon.max()):.1f}")

In [ ]:
import xarray as xr
import numpy as np

DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"

# Verificar as longitudes de um arquivo
ds = xr.open_dataset(f"{DIR_CMIP6}/tas_ssp245_2041_2060.nc", engine="netcdf4")
print("Lon CMIP6 original:", ds.lon.values[:5], "...", ds.lon.values[-5:])
print("Lat CMIP6:", ds.lat.values[:5], "...", ds.lat.values[-5:])

# Converter longitude
lon_conv = ds.lon.values - 360
print("Lon convertida:", lon_conv[:5], "...", lon_conv[-5:])

# Verificar se bate com ERA5
ds_clim = xr.open_dataset("/content/drive/MyDrive/era5_brasil/climatologia_completa_brasil.nc", engine="netcdf4")
print("\nLon ERA5:", ds_clim.lon.values[:5], "...", ds_clim.lon.values[-5:])
print("Lat ERA5:", ds_clim.lat.values[:5], "...", ds_clim.lat.values[-5:])

# Carregar tas com longitude corrigida e testar interpolação
tas = ds['tas']
tas_corr = tas.assign_coords(lon=lon_conv)
print("\ntas após correção lon:", tas_corr.lon.values[:3], "...", tas_corr.lon.values[-3:])

# Testar interpolação em um ponto
lat_era5 = ds_clim.lat.values
lon_era5 = ds_clim.lon.values

tas_clim = (tas_corr.mean('time') - 273.15)
print("\ntas_clim shape:", tas_clim.shape)
print("tas_clim valores (sample):", float(tas_clim.isel(lat=10, lon=10).values), "°C")

# Testar interp
try:
    tas_interp = tas_clim.interp(lat=lat_era5, lon=lon_era5, method='linear')
    print("Interpolação OK:", tas_interp.shape)
    print("Valores interpolados (sample):", float(tas_interp.isel(lat=100, lon=100).values), "°C")
    nan_pct = np.isnan(tas_interp.values).mean() * 100
    print(f"NaN: {nan_pct:.1f}%")
except Exception as e:
    print(f"Erro na interpolação: {e}")

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"
DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"

ds_clim     = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc",  engine="netcdf4")
ds_apt_atual = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc",         engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

CULTURAS = {
    "Lúpulo":   {"tmean_anual":(12,14,22,25),"tmin_min":(-20,-15,7,10),"tmax_max":(18,20,32,36),"precip_anual":(200,310,1370,1600),"chilling":(300,500,2000,2500)},
    "Pistache": {"tmean_anual":(15,18,35,38),"tmin_min":(-12,-10,7,10),"tmax_max":(25,30,45,48),"precip_anual":(150,250,600,800),  "chilling":(500,700,1500,1800)},
    "Avelã":    {"tmean_anual":(8,10,20,23), "tmin_min":(-18,-15,7,10),"tmax_max":(18,20,34,37),"precip_anual":(400,600,1200,1500),"chilling":(500,700,1800,2200)},
    "Amêndoa":  {"tmean_anual":(12,15,32,35),"tmin_min":(-6,-4,10,13), "tmax_max":(22,25,38,41),"precip_anual":(200,300,600,800),  "chilling":(150,250,600,800)},
    "Açafrão":  {"tmean_anual":(8,10,20,23), "tmin_min":(-12,-10,7,10),"tmax_max":(18,20,32,35),"precip_anual":(300,400,700,900),  "chilling":(400,600,1200,1500)},
    "Trufas":   {"tmean_anual":(8,10,20,23), "tmin_min":(-7,-5,7,10),  "tmax_max":(18,20,32,35),"precip_anual":(400,600,900,1100), "chilling":(400,600,1000,1300)},
}

def score_trap(val, a, b, c, d):
    return np.where(val<=a, 0., np.where(val<=b, (val-a)/(b-a),
           np.where(val<=c, 1., np.where(val<=d, (d-val)/(d-c), 0.)))).astype(float)

def calc_aptidao(req, tmean_a, tmin_m, tmax_m, precip_a, ch_a):
    return (score_trap(tmean_a,*req["tmean_anual"]) + score_trap(tmin_m,*req["tmin_min"]) +
            score_trap(tmax_m,*req["tmax_max"]) + score_trap(precip_a,*req["precip_anual"]) +
            score_trap(ch_a,*req["chilling"])) / 5

def load_interp(var, ssp, periodo):
    """Carrega variável CMIP6, corrige longitude e interpola para grid ERA5."""
    ds = xr.open_dataset(f"{DIR_CMIP6}/{var}_{ssp}_{periodo}.nc", engine="netcdf4")
    data_var = [v for v in ds.data_vars if v not in ['time_bnds','lat_bnds','lon_bnds']][0]
    da = ds[data_var].assign_coords(lon=ds.lon.values - 360)

    # Interpolar com extrapolação nas bordas
    da_interp = da.interp(lat=lat_era5, lon=lon_era5, method='linear',
                          kwargs={"fill_value": "extrapolate"})
    return da_interp

def processar_cenario(ssp, periodo):
    tas    = load_interp("tas",    ssp, periodo)
    tasmin = load_interp("tasmin", ssp, periodo)
    tasmax = load_interp("tasmax", ssp, periodo)
    pr     = load_interp("pr",     ssp, periodo)

    # Climatologia futura (média dos 20 anos)
    tas_clim    = (tas.mean('time')    - 273.15).values
    tasmin_clim = (tasmin.groupby('time.month').mean('time').min('month') - 273.15).values
    tasmax_clim = (tasmax.groupby('time.month').mean('time').max('month') - 273.15).values
    pr_clim     = (pr.mean('time') * 86400 * 30 * 12).values  # mm/ano

    # Baseline ERA5
    tmean_base  = ds_clim['tmean'].mean('month').values
    tmin_base   = ds_clim['tmin'].min('month').values
    tmax_base   = ds_clim['tmax'].max('month').values
    precip_base = ds_clim['precip'].sum('month').values
    ch_base     = ds_clim['chilling_hours_anual'].values

    # Delta method bias correction
    delta_t     = tas_clim - tmean_base
    delta_tmin  = tasmin_clim - tmin_base
    delta_tmax  = tasmax_clim - tmax_base
    ratio_pr    = np.where(precip_base > 10, pr_clim / (precip_base + 1e-6), 1.0)
    ratio_pr    = np.clip(ratio_pr, 0.5, 2.0)  # limitar a ±50% de variação

    tmean_fut  = tmean_base  + delta_t
    tmin_fut   = tmin_base   + delta_tmin
    tmax_fut   = tmax_base   + delta_tmax
    precip_fut = precip_base * ratio_pr
    ch_fut     = np.maximum(ch_base - delta_t * 50, 0)

    print(f"  ΔT médio: {delta_t.mean():.2f}°C | ΔTmin: {delta_tmin.mean():.2f}°C | Ratio precip: {ratio_pr.mean():.2f}")

    resultados = {}
    for cultura, req in CULTURAS.items():
        apt = calc_aptidao(req, tmean_fut, tmin_fut, tmax_fut, precip_fut, ch_fut)
        resultados[cultura] = apt
    return resultados

# Processar todos os cenários
print("Calculando aptidão futura...\n")
cenarios = {}
for ssp in ["ssp245", "ssp585"]:
    for periodo in ["2041_2060", "2071_2090"]:
        key = f"{ssp}_{periodo}"
        print(f"→ {key}")
        cenarios[key] = processar_cenario(ssp, periodo)

# Tabela de resultados
print("\n=== MUDANÇA DE APTIDÃO — SUL DO BRASIL (<-20°) ===")
mascara_sul = lat_era5 < -20

print(f"\n{'Cultura':<12} {'Atual':>7} {'SSP245-2050':>12} {'SSP585-2050':>12} {'SSP245-2080':>12} {'SSP585-2080':>12}")
print("-" * 67)

for cultura in CULTURAS:
    apt_atual = ds_apt_atual[cultura].values
    atual_pct = np.nanmean(apt_atual[mascara_sul, :] > 0.5) * 100
    row = f"{cultura:<12} {atual_pct:>6.1f}%"
    for key in ["ssp245_2041_2060","ssp585_2041_2060","ssp245_2071_2090","ssp585_2071_2090"]:
        apt_fut = cenarios[key][cultura]
        fut_pct = np.nanmean(apt_fut[mascara_sul, :] > 0.5) * 100
        delta = fut_pct - atual_pct
        row += f"  {fut_pct:>5.1f}%({'+' if delta>=0 else ''}{delta:.1f})"
    print(row)

# Salvar resultados
import xarray as xr
for key, res in cenarios.items():
    ds_out = xr.Dataset({
        c: xr.DataArray(v, dims=['lat','lon'], coords={'lat':lat_era5,'lon':lon_era5})
        for c, v in res.items()
    })
    ds_out.to_netcdf(f"{DIR_ERA5}/aptidao_{key}.nc")
    print(f"✓ Salvo: aptidao_{key}.nc")

In [ ]:
import xarray as xr
import numpy as np

DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"
DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"

ds_clim  = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

# Carregar tas e verificar passo a passo
ds = xr.open_dataset(f"{DIR_CMIP6}/tas_ssp245_2041_2060.nc", engine="netcdf4")
tas = ds['tas'].assign_coords(lon=ds.lon.values - 360)

print("CMIP6 lat range:", float(tas.lat.min()), "a", float(tas.lat.max()))
print("CMIP6 lon range:", float(tas.lon.min()), "a", float(tas.lon.max()))
print("ERA5  lat range:", lat_era5.min(), "a", lat_era5.max())
print("ERA5  lon range:", lon_era5.min(), "a", lon_era5.max())

# Média temporal primeiro
tas_mean = tas.mean('time') - 273.15
print("\ntas_mean shape:", tas_mean.shape)
print("tas_mean NaN:", int(np.isnan(tas_mean.values).sum()))
print("tas_mean sample:", float(tas_mean.isel(lat=5, lon=5).values))

# Interpolar
tas_interp = tas_mean.interp(lat=lat_era5, lon=lon_era5, method='linear',
                              kwargs={"fill_value": "extrapolate"})
print("\ntas_interp shape:", tas_interp.shape)
print("tas_interp NaN:", int(np.isnan(tas_interp.values).sum()))
print("tas_interp sample [100,100]:", float(tas_interp.isel(lat=100, lon=100).values))

# Comparar com ERA5 no mesmo ponto
tmean_base = ds_clim['tmean'].mean('month').values
print("ERA5 tmean [100,100]:", tmean_base[100, 100])
print("Delta T [100,100]:", float(tas_interp.isel(lat=100, lon=100).values) - tmean_base[100, 100])
print("Delta T médio (sem NaN):", np.nanmean(tas_interp.values - tmean_base))

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"
DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"

ds_clim      = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
ds_apt_atual = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc",         engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

# Baseline ERA5
tmean_base  = ds_clim['tmean'].mean('month').values
tmin_base   = ds_clim['tmin'].min('month').values
tmax_base   = ds_clim['tmax'].max('month').values
precip_base = ds_clim['precip'].sum('month').values
ch_base     = ds_clim['chilling_hours_anual'].values

CULTURAS = {
    "Lúpulo":   {"tmean_anual":(12,14,22,25),"tmin_min":(-20,-15,7,10),"tmax_max":(18,20,32,36),"precip_anual":(200,310,1370,1600),"chilling":(300,500,2000,2500)},
    "Pistache": {"tmean_anual":(15,18,35,38),"tmin_min":(-12,-10,7,10),"tmax_max":(25,30,45,48),"precip_anual":(150,250,600,800),  "chilling":(500,700,1500,1800)},
    "Avelã":    {"tmean_anual":(8,10,20,23), "tmin_min":(-18,-15,7,10),"tmax_max":(18,20,34,37),"precip_anual":(400,600,1200,1500),"chilling":(500,700,1800,2200)},
    "Amêndoa":  {"tmean_anual":(12,15,32,35),"tmin_min":(-6,-4,10,13), "tmax_max":(22,25,38,41),"precip_anual":(200,300,600,800),  "chilling":(150,250,600,800)},
    "Açafrão":  {"tmean_anual":(8,10,20,23), "tmin_min":(-12,-10,7,10),"tmax_max":(18,20,32,35),"precip_anual":(300,400,700,900),  "chilling":(400,600,1200,1500)},
    "Trufas":   {"tmean_anual":(8,10,20,23), "tmin_min":(-7,-5,7,10),  "tmax_max":(18,20,32,35),"precip_anual":(400,600,900,1100), "chilling":(400,600,1000,1300)},
}

def score_trap(val, a, b, c, d):
    return np.where(val<=a, 0., np.where(val<=b, (val-a)/(b-a),
           np.where(val<=c, 1., np.where(val<=d, (d-val)/(d-c), 0.)))).astype(float)

def calc_aptidao(req, tm, tn, tx, pr, ch):
    return (score_trap(tm,*req["tmean_anual"]) + score_trap(tn,*req["tmin_min"]) +
            score_trap(tx,*req["tmax_max"])    + score_trap(pr,*req["precip_anual"]) +
            score_trap(ch,*req["chilling"])) / 5

def interp_cmip6(var, ssp, periodo):
    """Carrega, corrige longitude e interpola para grid ERA5 — retorna numpy array."""
    ds = xr.open_dataset(f"{DIR_CMIP6}/{var}_{ssp}_{periodo}.nc", engine="netcdf4")
    dv = [v for v in ds.data_vars if v not in ['time_bnds','lat_bnds','lon_bnds']][0]
    da = ds[dv].assign_coords(lon=ds.lon.values - 360)
    da_interp = da.interp(lat=lat_era5, lon=lon_era5, method='linear',
                          kwargs={"fill_value": "extrapolate"})
    return da_interp  # manter como DataArray para operações mensais

def processar_cenario(ssp, periodo):
    tas    = interp_cmip6("tas",    ssp, periodo)
    tasmin = interp_cmip6("tasmin", ssp, periodo)
    tasmax = interp_cmip6("tasmax", ssp, periodo)
    pr     = interp_cmip6("pr",     ssp, periodo)

    # Climatologia futura em numpy
    tas_fut    = (tas.mean('time')    - 273.15).values
    tasmin_fut = (tasmin.groupby('time.month').mean('time').min('month') - 273.15).values
    tasmax_fut = (tasmax.groupby('time.month').mean('time').max('month') - 273.15).values
    pr_fut     = (pr.mean('time') * 86400 * 30 * 12).values  # mm/ano

    # Delta method
    delta_t   = tas_fut    - tmean_base
    delta_tn  = tasmin_fut - tmin_base
    delta_tx  = tasmax_fut - tmax_base
    ratio_pr  = np.clip(pr_fut / np.where(precip_base > 10, precip_base, 10), 0.5, 2.0)

    tmean_fut  = tmean_base  + delta_t
    tmin_fut   = tmin_base   + delta_tn
    tmax_fut   = tmax_base   + delta_tx
    precip_fut = precip_base * ratio_pr
    ch_fut     = np.maximum(ch_base - delta_t * 50, 0)

    print(f"  ΔT={np.nanmean(delta_t):.2f}°C | ΔTmin={np.nanmean(delta_tn):.2f}°C | ΔTmax={np.nanmean(delta_tx):.2f}°C | ratio_pr={np.nanmean(ratio_pr):.2f}")

    return {c: calc_aptidao(req, tmean_fut, tmin_fut, tmax_fut, precip_fut, ch_fut)
            for c, req in CULTURAS.items()}

# Processar todos os cenários
print("Calculando aptidão futura...\n")
cenarios = {}
for ssp in ["ssp245", "ssp585"]:
    for periodo in ["2041_2060", "2071_2090"]:
        key = f"{ssp}_{periodo}"
        print(f"→ {key}")
        cenarios[key] = processar_cenario(ssp, periodo)

# Tabela de resultados
print("\n=== MUDANÇA DE APTIDÃO — SUL DO BRASIL (<-20°) ===")
mascara_sul = lat_era5 < -20

print(f"\n{'Cultura':<12} {'Atual':>7} {'SSP245-2050':>12} {'SSP585-2050':>12} {'SSP245-2080':>12} {'SSP585-2080':>12}")
print("-" * 67)

for cultura in CULTURAS:
    apt_atual = ds_apt_atual[cultura].values
    atual_pct = np.nanmean(apt_atual[mascara_sul,:] > 0.5) * 100
    row = f"{cultura:<12} {atual_pct:>6.1f}%"
    for key in ["ssp245_2041_2060","ssp585_2041_2060","ssp245_2071_2090","ssp585_2071_2090"]:
        apt_fut = cenarios[key][cultura]
        fut_pct = np.nanmean(apt_fut[mascara_sul,:] > 0.5) * 100
        delta   = fut_pct - atual_pct
        row += f"  {fut_pct:>5.1f}%({'+' if delta>=0 else ''}{delta:.1f})"
    print(row)

# Salvar
for key, res in cenarios.items():
    ds_out = xr.Dataset({c: xr.DataArray(v, dims=['lat','lon'],
                         coords={'lat':lat_era5,'lon':lon_era5})
                         for c, v in res.items()})
    ds_out.to_netcdf(f"{DIR_ERA5}/aptidao_{key}.nc")
print("\n✓ Arquivos salvos!")

In [ ]:
import xarray as xr
import numpy as np
import os

DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"
DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"

ds_clim      = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
ds_apt_atual = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc",         engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

tmean_base  = ds_clim['tmean'].mean('month').values
tmin_base   = ds_clim['tmin'].min('month').values
tmax_base   = ds_clim['tmax'].max('month').values
precip_base = ds_clim['precip'].sum('month').values
ch_base     = ds_clim['chilling_hours_anual'].values

CULTURAS = {
    "Lúpulo":   {"tmean_anual":(12,14,22,25),"tmin_min":(-20,-15,7,10),"tmax_max":(18,20,32,36),"precip_anual":(200,310,1370,1600),"chilling":(300,500,2000,2500)},
    "Pistache": {"tmean_anual":(15,18,35,38),"tmin_min":(-12,-10,7,10),"tmax_max":(25,30,45,48),"precip_anual":(150,250,600,800),  "chilling":(500,700,1500,1800)},
    "Avelã":    {"tmean_anual":(8,10,20,23), "tmin_min":(-18,-15,7,10),"tmax_max":(18,20,34,37),"precip_anual":(400,600,1200,1500),"chilling":(500,700,1800,2200)},
    "Amêndoa":  {"tmean_anual":(12,15,32,35),"tmin_min":(-6,-4,10,13), "tmax_max":(22,25,38,41),"precip_anual":(200,300,600,800),  "chilling":(150,250,600,800)},
    "Açafrão":  {"tmean_anual":(8,10,20,23), "tmin_min":(-12,-10,7,10),"tmax_max":(18,20,32,35),"precip_anual":(300,400,700,900),  "chilling":(400,600,1200,1500)},
    "Trufas":   {"tmean_anual":(8,10,20,23), "tmin_min":(-7,-5,7,10),  "tmax_max":(18,20,32,35),"precip_anual":(400,600,900,1100), "chilling":(400,600,1000,1300)},
}

def score_trap(val, a, b, c, d):
    return np.where(val<=a, 0., np.where(val<=b, (val-a)/(b-a),
           np.where(val<=c, 1., np.where(val<=d, (d-val)/(d-c), 0.)))).astype(float)

def calc_aptidao(req, tm, tn, tx, pr, ch):
    return (score_trap(tm,*req["tmean_anual"]) + score_trap(tn,*req["tmin_min"]) +
            score_trap(tx,*req["tmax_max"])    + score_trap(pr,*req["precip_anual"]) +
            score_trap(ch,*req["chilling"])) / 5

def processar_cenario(ssp, periodo):
    # Carregar apenas tas e pr (mais confiáveis no CMIP6 mensal)
    ds_tas = xr.open_dataset(f"{DIR_CMIP6}/tas_{ssp}_{periodo}.nc",    engine="netcdf4")
    ds_pr  = xr.open_dataset(f"{DIR_CMIP6}/pr_{ssp}_{periodo}.nc",     engine="netcdf4")

    tas = ds_tas['tas'].assign_coords(lon=ds_tas.lon.values - 360)
    pr  = ds_pr['pr'].assign_coords(lon=ds_pr.lon.values - 360)

    # Interpolar para grid ERA5
    tas_interp = tas.interp(lat=lat_era5, lon=lon_era5, method='linear',
                             kwargs={"fill_value":"extrapolate"})
    pr_interp  = pr.interp(lat=lat_era5, lon=lon_era5, method='linear',
                            kwargs={"fill_value":"extrapolate"})

    # ΔT por mês (preserva sazonalidade)
    tas_mensal_fut  = tas_interp.groupby('time.month').mean('time') - 273.15  # (12, lat, lon)
    pr_mensal_fut   = pr_interp.groupby('time.month').mean('time') * 86400 * 30  # mm/mês

    # Baseline ERA5 mensal
    tmean_mensal_base = ds_clim['tmean']  # (12, lat, lon)

    # Delta mensal de temperatura
    delta_t_mensal = tas_mensal_fut.values - tmean_mensal_base.values  # (12, lat, lon)
    delta_t_anual  = delta_t_mensal.mean(axis=0)  # média anual do delta

    # Aplicar delta nas variáveis derivadas
    tmean_fut  = tmean_base  + delta_t_anual
    tmin_fut   = tmin_base   + delta_t_anual   # mesmo delta para Tmin
    tmax_fut   = tmax_base   + delta_t_anual   # mesmo delta para Tmax

    # Precipitação: ratio mensal
    precip_mensal_base = ds_clim['precip'].values  # (12, lat, lon)
    ratio_pr = np.where(precip_mensal_base > 5,
                        pr_mensal_fut.values / (precip_mensal_base + 1e-6), 1.0)
    ratio_pr = np.clip(ratio_pr, 0.3, 3.0)
    precip_fut = (precip_mensal_base * ratio_pr).sum(axis=0)  # mm/ano

    # Chilling hours: reduz proporcionalmente ao aquecimento
    ch_fut = np.maximum(ch_base - delta_t_anual * 50, 0)

    print(f"  ΔT anual médio: {delta_t_anual.mean():.2f}°C | Precip ratio: {ratio_pr.mean():.2f}")
    print(f"  Tmean fut Sul: {tmean_fut[lat_era5<-20,:].mean():.1f}°C (atual: {tmean_base[lat_era5<-20,:].mean():.1f}°C)")

    return {c: calc_aptidao(req, tmean_fut, tmin_fut, tmax_fut, precip_fut, ch_fut)
            for c, req in CULTURAS.items()}

print("Calculando aptidão futura...\n")
cenarios = {}
for ssp in ["ssp245", "ssp585"]:
    for periodo in ["2041_2060", "2071_2090"]:
        key = f"{ssp}_{periodo}"
        print(f"→ {key}")
        cenarios[key] = processar_cenario(ssp, periodo)

# Tabela
print("\n=== MUDANÇA DE APTIDÃO — SUL DO BRASIL (<-20°) ===")
mascara_sul = lat_era5 < -20
print(f"\n{'Cultura':<12} {'Atual':>7} {'SSP245-2050':>12} {'SSP585-2050':>12} {'SSP245-2080':>12} {'SSP585-2080':>12}")
print("-" * 67)

for cultura in CULTURAS:
    apt_atual = ds_apt_atual[cultura].values
    atual_pct = np.nanmean(apt_atual[mascara_sul,:] > 0.5) * 100
    row = f"{cultura:<12} {atual_pct:>6.1f}%"
    for key in ["ssp245_2041_2060","ssp585_2041_2060","ssp245_2071_2090","ssp585_2071_2090"]:
        apt_fut = cenarios[key][cultura]
        fut_pct = np.nanmean(apt_fut[mascara_sul,:] > 0.5) * 100
        delta   = fut_pct - atual_pct
        row += f"  {fut_pct:>5.1f}%({'+' if delta>=0 else ''}{delta:.1f})"
    print(row)

# Salvar
for key, res in cenarios.items():
    ds_out = xr.Dataset({c: xr.DataArray(v, dims=['lat','lon'],
                         coords={'lat':lat_era5,'lon':lon_era5})
                         for c, v in res.items()})
    ds_out.to_netcdf(f"{DIR_ERA5}/aptidao_{key}.nc")
print("\n✓ Salvos!")

In [ ]:
import xarray as xr
import numpy as np

DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"
DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"

ds_clim  = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

ds_tas = xr.open_dataset(f"{DIR_CMIP6}/tas_ssp245_2041_2060.nc", engine="netcdf4")
tas = ds_tas['tas'].assign_coords(lon=ds_tas.lon.values - 360)
tas_interp = tas.interp(lat=lat_era5, lon=lon_era5, method='linear',
                         kwargs={"fill_value":"extrapolate"})

tas_mensal_fut = tas_interp.groupby('time.month').mean('time') - 273.15
print("tas_mensal_fut shape:", tas_mensal_fut.shape)
print("tas_mensal_fut dims:", tas_mensal_fut.dims)
print("tas_mensal_fut NaN:", int(np.isnan(tas_mensal_fut.values).sum()))
print("tas_mensal_fut sample:", float(tas_mensal_fut.isel(month=0, lat=100, lon=100).values))

tmean_mensal_base = ds_clim['tmean']
print("\ntmean_mensal_base shape:", tmean_mensal_base.shape)
print("tmean_mensal_base dims:", tmean_mensal_base.dims)
print("tmean_mensal_base NaN:", int(np.isnan(tmean_mensal_base.values).sum()))
print("tmean_mensal_base sample:", float(tmean_mensal_base.isel(month=0, lat=100, lon=100).values))

# Tentar subtração alinhando pelo xarray
delta = tas_mensal_fut - tmean_mensal_base
print("\ndelta shape:", delta.shape)
print("delta NaN:", int(np.isnan(delta.values).sum()))
print("delta sample:", float(delta.isel(month=0, lat=100, lon=100).values))
print("delta médio:", float(delta.mean().values))

In [ ]:
import xarray as xr
import numpy as np
import os

DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"
DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"

ds_clim      = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
ds_apt_atual = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc",         engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

tmean_base  = ds_clim['tmean'].mean('month').values
tmin_base   = ds_clim['tmin'].min('month').values
tmax_base   = ds_clim['tmax'].max('month').values
precip_base = ds_clim['precip'].sum('month').values
ch_base     = ds_clim['chilling_hours_anual'].values

CULTURAS = {
    "Lúpulo":   {"tmean_anual":(12,14,22,25),"tmin_min":(-20,-15,7,10),"tmax_max":(18,20,32,36),"precip_anual":(200,310,1370,1600),"chilling":(300,500,2000,2500)},
    "Pistache": {"tmean_anual":(15,18,35,38),"tmin_min":(-12,-10,7,10),"tmax_max":(25,30,45,48),"precip_anual":(150,250,600,800),  "chilling":(500,700,1500,1800)},
    "Avelã":    {"tmean_anual":(8,10,20,23), "tmin_min":(-18,-15,7,10),"tmax_max":(18,20,34,37),"precip_anual":(400,600,1200,1500),"chilling":(500,700,1800,2200)},
    "Amêndoa":  {"tmean_anual":(12,15,32,35),"tmin_min":(-6,-4,10,13), "tmax_max":(22,25,38,41),"precip_anual":(200,300,600,800),  "chilling":(150,250,600,800)},
    "Açafrão":  {"tmean_anual":(8,10,20,23), "tmin_min":(-12,-10,7,10),"tmax_max":(18,20,32,35),"precip_anual":(300,400,700,900),  "chilling":(400,600,1200,1500)},
    "Trufas":   {"tmean_anual":(8,10,20,23), "tmin_min":(-7,-5,7,10),  "tmax_max":(18,20,32,35),"precip_anual":(400,600,900,1100), "chilling":(400,600,1000,1300)},
}

def score_trap(val, a, b, c, d):
    return np.where(val<=a, 0., np.where(val<=b, (val-a)/(b-a),
           np.where(val<=c, 1., np.where(val<=d, (d-val)/(d-c), 0.)))).astype(float)

def calc_aptidao(req, tm, tn, tx, pr, ch):
    return (score_trap(tm,*req["tmean_anual"]) + score_trap(tn,*req["tmin_min"]) +
            score_trap(tx,*req["tmax_max"])    + score_trap(pr,*req["precip_anual"]) +
            score_trap(ch,*req["chilling"])) / 5

def processar_cenario(ssp, periodo):
    ds_tas = xr.open_dataset(f"{DIR_CMIP6}/tas_{ssp}_{periodo}.nc", engine="netcdf4")
    ds_pr  = xr.open_dataset(f"{DIR_CMIP6}/pr_{ssp}_{periodo}.nc",  engine="netcdf4")

    tas = ds_tas['tas'].assign_coords(lon=ds_tas.lon.values - 360)
    pr  = ds_pr['pr'].assign_coords(lon=ds_pr.lon.values - 360)

    tas_interp = tas.interp(lat=lat_era5, lon=lon_era5, method='linear',
                             kwargs={"fill_value":"extrapolate"})
    pr_interp  = pr.interp(lat=lat_era5, lon=lon_era5, method='linear',
                            kwargs={"fill_value":"extrapolate"})

    # Climatologia futura mensal — tudo em numpy daqui em diante
    tas_mensal = (tas_interp.groupby('time.month').mean('time') - 273.15).values  # (12,401,461)
    pr_mensal  = (pr_interp.groupby('time.month').mean('time') * 86400 * 30).values

    # Baseline mensal
    tmean_mensal_base = ds_clim['tmean'].values  # (12,401,461) com NaN no oceano

    # Delta mensal — NaN onde não há terra (correto, não propaga)
    delta_t_mensal = tas_mensal - tmean_mensal_base   # (12,401,461)
    delta_t_anual  = np.nanmean(delta_t_mensal, axis=0)  # (401,461) — nanmean ignora NaN

    # Aplicar delta
    tmean_fut  = tmean_base + delta_t_anual
    tmin_fut   = tmin_base  + delta_t_anual
    tmax_fut   = tmax_base  + delta_t_anual
    ch_fut     = np.maximum(ch_base - delta_t_anual * 50, 0)

    # Precipitação
    precip_mensal_base = ds_clim['precip'].values  # (12,401,461)
    ratio_pr = np.where(precip_mensal_base > 5,
                        pr_mensal / (precip_mensal_base + 1e-6), 1.0)
    ratio_pr   = np.clip(ratio_pr, 0.3, 3.0)
    precip_fut = np.nansum(precip_mensal_base * ratio_pr, axis=0)

    # Estatísticas só em pixels terrestres (não NaN)
    mascara_terra = ~np.isnan(tmean_base)
    print(f"  ΔT médio terra: {np.nanmean(delta_t_anual[mascara_terra]):.2f}°C")
    print(f"  Tmean fut Sul:  {np.nanmean(tmean_fut[lat_era5<-20,:]):.1f}°C  (base: {np.nanmean(tmean_base[lat_era5<-20,:]):.1f}°C)")

    return {c: calc_aptidao(req, tmean_fut, tmin_fut, tmax_fut, precip_fut, ch_fut)
            for c, req in CULTURAS.items()}

print("Calculando aptidão futura...\n")
cenarios = {}
for ssp in ["ssp245", "ssp585"]:
    for periodo in ["2041_2060", "2071_2090"]:
        key = f"{ssp}_{periodo}"
        print(f"→ {key}")
        cenarios[key] = processar_cenario(ssp, periodo)

print("\n=== MUDANÇA DE APTIDÃO — SUL DO BRASIL (<-20°) ===")
mascara_sul = lat_era5 < -20
print(f"\n{'Cultura':<12} {'Atual':>7} {'SSP245-2050':>12} {'SSP585-2050':>12} {'SSP245-2080':>12} {'SSP585-2080':>12}")
print("-" * 67)

for cultura in CULTURAS:
    apt_atual = ds_apt_atual[cultura].values
    atual_pct = np.nanmean(apt_atual[mascara_sul,:] > 0.5) * 100
    row = f"{cultura:<12} {atual_pct:>6.1f}%"
    for key in ["ssp245_2041_2060","ssp585_2041_2060","ssp245_2071_2090","ssp585_2071_2090"]:
        apt_fut = cenarios[key][cultura]
        fut_pct = np.nanmean(apt_fut[mascara_sul,:] > 0.5) * 100
        delta   = fut_pct - atual_pct
        row += f"  {fut_pct:>5.1f}%({'+' if delta>=0 else ''}{delta:.1f})"
    print(row)

# Salvar
for key, res in cenarios.items():
    ds_out = xr.Dataset({c: xr.DataArray(v, dims=['lat','lon'],
                         coords={'lat':lat_era5,'lon':lon_era5})
                         for c, v in res.items()})
    ds_out.to_netcdf(f"{DIR_ERA5}/aptidao_{key}.nc")
print("\n✓ Salvos!")

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

DIR_ERA5 = "/content/drive/MyDrive/era5_brasil"

ds_apt_atual = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc", engine="netcdf4")
lat = ds_apt_atual['lat'].values
lon = ds_apt_atual['lon'].values

# Carregar aptidão futura
cenarios_ds = {}
for ssp in ["ssp245", "ssp585"]:
    for periodo in ["2041_2060", "2071_2090"]:
        key = f"{ssp}_{periodo}"
        cenarios_ds[key] = xr.open_dataset(f"{DIR_ERA5}/aptidao_{key}.nc", engine="netcdf4")

# Mapa de GANHO/PERDA para cada cultura
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Usar SSP585 2071-2090 (cenário mais extremo — mais dramático visualmente)
key_plot = "ssp585_2071_2090"

cmap_delta = plt.cm.RdBu
norm_delta = mcolors.TwoSlopeNorm(vmin=-0.5, vcenter=0, vmax=0.5)

for i, cultura in enumerate(["Lúpulo","Pistache","Avelã","Amêndoa","Açafrão","Trufas"]):
    ax = axes[i]
    apt_atual = ds_apt_atual[cultura].values
    apt_fut   = cenarios_ds[key_plot][cultura].values
    delta     = apt_fut - apt_atual

    im = ax.pcolormesh(lon, lat, delta, cmap=cmap_delta, norm=norm_delta)
    ax.set_title(f"{cultura}\n(SSP5-8.5, 2071–2090)", fontsize=11, fontweight='bold')
    ax.axhline(-20, color='gray', linewidth=0.7, linestyle='--')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    plt.colorbar(im, ax=ax, label='ΔAptidão')

    # Estatísticas rápidas
    ganho = np.nansum((apt_fut > 0.5) & (apt_atual <= 0.5))
    perda = np.nansum((apt_fut <= 0.5) & (apt_atual > 0.5))
    ax.set_title(f"{cultura} | +{ganho} px ganho / -{perda} px perda", fontsize=10, fontweight='bold')

plt.suptitle('Mudança de Aptidão Climática — SSP5-8.5 (2071–2090) vs 1991–2020',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{DIR_ERA5}/delta_aptidao_ssp585_2080.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Mapa de delta salvo!")

# Verificar Nordeste para pistache
ds_clim = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
lat_arr = ds_clim['lat'].values
mascara_ne = (lat_arr > -15) & (lat_arr < -3)

apt_pistache_atual = ds_apt_atual['Pistache'].values
apt_pistache_fut   = cenarios_ds["ssp585_2071_2090"]['Pistache'].values

print("\n=== PISTACHE NO NORDESTE (<-3° a >-15°) ===")
print(f"  Atual:  {np.nanmean(apt_pistache_atual[mascara_ne,:] > 0.5)*100:.1f}% apto")
print(f"  2080:   {np.nanmean(apt_pistache_fut[mascara_ne,:] > 0.5)*100:.1f}% apto")

In [ ]:
import xarray as xr
import numpy as np

DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"
DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"

ds_clim = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

# Carregar também o histórico do MIROC6 para comparar
# Primeiro verificar o que temos — o baseline ERA5 é 1991-2020
# O CMIP6 futuro é 2041-2060 e 2071-2090
# O delta deveria ser calculado em relação ao período histórico do MIROC6, não do ERA5!

# Verificar temperatura média do MIROC6 futuro vs ERA5
for ssp in ["ssp245", "ssp585"]:
    for periodo in ["2041_2060", "2071_2090"]:
        ds = xr.open_dataset(f"{DIR_CMIP6}/tas_{ssp}_{periodo}.nc", engine="netcdf4")
        tas = ds['tas'].assign_coords(lon=ds.lon.values - 360)
        tas_interp = tas.interp(lat=lat_era5, lon=lon_era5, method='linear',
                                kwargs={"fill_value":"extrapolate"})
        tas_mean = (tas_interp.mean('time') - 273.15).values
        era5_mean = ds_clim['tmean'].mean('month').values
        delta = np.nanmean(tas_mean - era5_mean)
        print(f"{ssp} {periodo}: MIROC6_fut={np.nanmean(tas_mean):.2f}°C | ERA5_base={np.nanmean(era5_mean):.2f}°C | ΔT={delta:.2f}°C")

print()
# O problema: estamos comparando MIROC6 futuro com ERA5 baseline
# O correto é: MIROC6 futuro - MIROC6 histórico (mesmo modelo)
# Precisamos baixar o período histórico do MIROC6

# Verificar se o delta faz mais sentido olhando só para o Brasil
print("\nVerificação por região:")
for ssp in ["ssp245"]:
    ds = xr.open_dataset(f"{DIR_CMIP6}/tas_{ssp}_2041_2060.nc", engine="netcdf4")
    tas = ds['tas'].assign_coords(lon=ds.lon.values - 360)
    # Média direta sem interpolar — valor bruto do MIROC6
    tas_brasil_mean = float((ds['tas'].mean('time') - 273.15).mean().values)
    print(f"MIROC6 {ssp} 2041-2060 temperatura média bruta: {tas_brasil_mean:.2f}°C")

# ERA5 temperatura média atual
era5_tmean = np.nanmean(ds_clim['tmean'].mean('month').values)
print(f"ERA5 temperatura média atual (1991-2020): {era5_tmean:.2f}°C")
print(f"\nDiagnóstico: se MIROC6_fut >> ERA5_base, o problema é bias do modelo, não aquecimento real")
print("Solução correta: baixar MIROC6 histórico (1991-2020) e calcular delta = futuro - histórico")

In [ ]:
import requests

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

payload_hist = {
    "model": "miroc6",
    "experiment": "historical",
    "variable": "near_surface_air_temperature",
    "temporal_resolution": "monthly",
    "year": [str(a) for a in range(1991, 2015)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "area": [6, -74, -34, -28],
    "data_format": "netcdf",
}

r = requests.post(
    f"{BASE}/retrieve/v1/processes/projections-cmip6/execution",
    json={"inputs": payload_hist}, headers=headers)
print("Status:", r.status_code)
print("Job ID:", r.json().get("jobID"))

In [ ]:
import requests, time, zipfile, os, shutil
from google.colab import drive
drive.mount('/content/drive')

KEY  = "dc7206be-f2ee-4113-a186-ea9d36afc0bd"
BASE = "https://cds.climate.copernicus.eu/api"
headers = {"PRIVATE-TOKEN": KEY, "Content-Type": "application/json"}

DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"
job_id = "60d3cd23-febe-4cdd-9edd-acdacfa6ab4c"

print("Aguardando histórico MIROC6...")
for _ in range(120):
    r = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}", headers=headers)
    status = r.json().get("status")
    print(f"  {status}")
    if status == "successful":
        r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
        url = r2.json()["asset"]["value"]["href"]
        size = r2.json()["asset"]["value"].get("file:size", 0)
        print(f"  Tamanho: {size/1e6:.1f} MB")
        tmp = "/tmp/miroc6_hist.zip"
        r3 = requests.get(url, stream=True)
        with open(tmp, "wb") as f:
            for chunk in r3.iter_content(65536): f.write(chunk)
        with zipfile.ZipFile(tmp) as z:
            names = [n for n in z.namelist() if n.endswith(".nc")]
            print("  Arquivos NC:", names)
            z.extractall("/tmp/miroc6_hist/")
        nc_dst = f"{DIR_CMIP6}/tas_historical_1991_2014.nc"
        shutil.copy2(f"/tmp/miroc6_hist/{names[0]}", nc_dst)
        os.remove(tmp)
        print(f"  ✓ Salvo: {nc_dst} ({os.path.getsize(nc_dst)/1e6:.1f} MB)")
        break
    elif status == "failed":
        r2 = requests.get(f"{BASE}/retrieve/v1/jobs/{job_id}/results", headers=headers)
        print(f"  ✗ {r2.json().get('traceback','')[:200]}")
        break
    time.sleep(20)

In [ ]:
import xarray as xr
import numpy as np

DIR_ERA5  = "/content/drive/MyDrive/era5_brasil"
DIR_CMIP6 = "/content/drive/MyDrive/era5_brasil/cmip6"

ds_clim = xr.open_dataset(f"{DIR_ERA5}/climatologia_completa_brasil.nc", engine="netcdf4")
lat_era5 = ds_clim['lat'].values
lon_era5 = ds_clim['lon'].values

# Carregar histórico MIROC6
ds_hist = xr.open_dataset(f"{DIR_CMIP6}/tas_historical_1991_2014.nc", engine="netcdf4")
tas_hist = ds_hist['tas'].assign_coords(lon=ds_hist.lon.values - 360)
tas_hist_interp = tas_hist.interp(lat=lat_era5, lon=lon_era5, method='linear',
                                   kwargs={"fill_value":"extrapolate"})
miroc6_hist_mean = (tas_hist_interp.mean('time') - 273.15).values

# ERA5 baseline
era5_mean = ds_clim['tmean'].mean('month').values

# Bias espacial
bias = miroc6_hist_mean - era5_mean
print(f"Bias MIROC6 vs ERA5:")
print(f"  Média Brasil:  {np.nanmean(bias):.2f}°C")
print(f"  Bias Sul:      {np.nanmean(bias[lat_era5 < -20, :]):.2f}°C")
print(f"  Bias Norte:    {np.nanmean(bias[lat_era5 > 0, :]):.2f}°C")
print(f"  Min: {np.nanmin(bias):.2f}°C | Max: {np.nanmax(bias):.2f}°C")

# ΔT real por cenário (futuro - histórico, mesmo modelo)
print("\nΔT real (MIROC6 futuro - MIROC6 histórico):")
for ssp in ["ssp245", "ssp585"]:
    for periodo in ["2041_2060", "2071_2090"]:
        ds_fut = xr.open_dataset(f"{DIR_CMIP6}/tas_{ssp}_{periodo}.nc", engine="netcdf4")
        tas_fut = ds_fut['tas'].assign_coords(lon=ds_fut.lon.values - 360)
        tas_fut_interp = tas_fut.interp(lat=lat_era5, lon=lon_era5, method='linear',
                                         kwargs={"fill_value":"extrapolate"})
        miroc6_fut_mean = (tas_fut_interp.mean('time') - 273.15).values
        delta_real = miroc6_fut_mean - miroc6_hist_mean
        print(f"  {ssp} {periodo}: ΔT={np.nanmean(delta_real):.2f}°C | Sul: {np.nanmean(delta_real[lat_era5 < -20,:]):.2f}°C")

In [ ]:
import xarray as xr
import numpy as np

DIR_ERA5 = "/content/drive/MyDrive/era5_brasil"

for key, res in cenarios.items():
    out = f"{DIR_ERA5}/aptidao_{key}_v2.nc"
    ds_out = xr.Dataset({
        c: xr.DataArray(v, dims=['lat','lon'],
                        coords={'lat': lat_era5, 'lon': lon_era5})
        for c, v in res.items()
    })
    ds_out.to_netcdf(out, mode='w')
    print(f"✓ {out}")

print("\nArquivos salvos!")

In [ ]:
!pip install geopandas matplotlib cartopy -q

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import xarray as xr
import requests, zipfile, os

DIR_ERA5 = "/content/drive/MyDrive/era5_brasil"

# Baixar shapefile estados IBGE
if not os.path.exists("/tmp/estados_shp/"):
    print("Baixando shapefile estados...")
    r = requests.get("https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/Brasil/BR/BR_UF_2022.zip", stream=True)
    with open("/tmp/estados.zip", "wb") as f:
        for chunk in r.iter_content(8192*16): f.write(chunk)
    with zipfile.ZipFile("/tmp/estados.zip") as z:
        z.extractall("/tmp/estados_shp/")
    print("✓ Extraído")

import glob
shp_estados = glob.glob("/tmp/estados_shp/*.shp")[0]
estados = gpd.read_file(shp_estados)
print("Shapefile OK:", len(estados), "estados")

# Carregar dados
ds_apt_atual = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc",      engine="netcdf4")
ds_fut_245_2050 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp245_2041_2060_v2.nc", engine="netcdf4")
ds_fut_585_2080 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp585_2071_2090_v2.nc", engine="netcdf4")

lat = ds_apt_atual['lat'].values
lon = ds_apt_atual['lon'].values

CULTURAS = ["Lúpulo", "Pistache", "Avelã", "Amêndoa", "Açafrão", "Trufas"]

# ── FIGURA 1: Aptidão atual ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

cmap_apt = plt.cm.RdYlGn
norm_apt = mcolors.Normalize(vmin=0, vmax=1)

for i, cultura in enumerate(CULTURAS):
    ax = axes[i]
    apt = ds_apt_atual[cultura].values

    im = ax.pcolormesh(lon, lat, apt, cmap=cmap_apt, norm=norm_apt, rasterized=True)

    # Contorno dos estados
    estados.boundary.plot(ax=ax, linewidth=0.4, color='black', alpha=0.6)

    # Estatísticas
    apt_br = apt[~np.isnan(apt)]
    pct_apta = np.sum(apt_br > 0.5) / len(apt_br) * 100
    pct_otima = np.sum(apt_br > 0.7) / len(apt_br) * 100

    ax.set_title(f"{cultura}", fontsize=13, fontweight='bold', pad=4)
    ax.set_xlabel('Longitude', fontsize=9)
    ax.set_ylabel('Latitude', fontsize=9)
    ax.set_xlim(-74, -28); ax.set_ylim(-34, 6)
    ax.tick_params(labelsize=8)
    ax.text(0.02, 0.04, f"Apta: {pct_apta:.1f}% | Ótima: {pct_otima:.1f}%",
            transform=ax.transAxes, fontsize=8, color='black',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    cb = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cb.set_label('Índice de Aptidão (0–1)', fontsize=8)
    cb.ax.tick_params(labelsize=7)

plt.suptitle('Aptidão Climática Atual — Brasil (1991–2020)\nMétodo: Função trapezoidal multivariada',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f"{DIR_ERA5}/fig1_aptidao_atual.png", dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✓ Figura 1 salva!")

# ── FIGURA 2: Delta de aptidão SSP245-2050 vs SSP585-2080 ────────────────────
fig, axes = plt.subplots(3, 4, figsize=(22, 15))

cmap_d = plt.cm.RdBu_r
norm_d = mcolors.TwoSlopeNorm(vmin=-0.3, vcenter=0, vmax=0.3)

ds_fut_245_2080 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp245_2071_2090_v2.nc", engine="netcdf4")
ds_fut_585_2050 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp585_2041_2060_v2.nc", engine="netcdf4")

cenarios_plot = [
    (ds_fut_245_2050, "SSP2-4.5\n2041–2060"),
    (ds_fut_585_2050, "SSP5-8.5\n2041–2060"),
    (ds_fut_245_2080, "SSP2-4.5\n2071–2090"),
    (ds_fut_585_2080, "SSP5-8.5\n2071–2090"),
]

for row, cultura in enumerate(["Lúpulo", "Amêndoa", "Pistache"]):
    apt_atual = ds_apt_atual[cultura].values
    for col, (ds_fut, label) in enumerate(cenarios_plot):
        ax = axes[row, col]
        apt_fut = ds_fut[cultura].values
        delta   = apt_fut - apt_atual

        im = ax.pcolormesh(lon, lat, delta, cmap=cmap_d, norm=norm_d, rasterized=True)
        estados.boundary.plot(ax=ax, linewidth=0.3, color='black', alpha=0.5)

        if row == 0:
            ax.set_title(label, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f"{cultura}\nLatitude", fontsize=10, fontweight='bold')
        else:
            ax.set_ylabel('Latitude', fontsize=8)

        ax.set_xlabel('Longitude', fontsize=8)
        ax.set_xlim(-74, -28); ax.set_ylim(-34, 6)
        ax.tick_params(labelsize=7)

        ganho = int(np.nansum((apt_fut > 0.5) & (apt_atual <= 0.5)))
        perda = int(np.nansum((apt_fut <= 0.5) & (apt_atual > 0.5)))
        ax.text(0.02, 0.04, f"+{ganho} / -{perda} pixels",
                transform=ax.transAxes, fontsize=7,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

        cb = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
        cb.set_label('ΔAptidão', fontsize=7)
        cb.ax.tick_params(labelsize=6)

plt.suptitle('Mudança de Aptidão Climática por Cenário — Lúpulo, Amêndoa e Pistache\n(bias-corrected MIROC6)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f"{DIR_ERA5}/fig2_delta_aptidao.png", dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✓ Figura 2 salva!")

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import glob

DIR_ERA5 = "/content/drive/MyDrive/era5_brasil"

ds_apt_atual    = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc",           engine="netcdf4")
ds_fut_245_2050 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp245_2041_2060_v2.nc",   engine="netcdf4")
ds_fut_245_2080 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp245_2071_2090_v2.nc",   engine="netcdf4")
ds_fut_585_2050 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp585_2041_2060_v2.nc",   engine="netcdf4")
ds_fut_585_2080 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp585_2071_2090_v2.nc",   engine="netcdf4")

lat = ds_apt_atual['lat'].values
lon = ds_apt_atual['lon'].values

shp_estados = glob.glob("/tmp/estados_shp/*.shp")[0]
estados = gpd.read_file(shp_estados)

CULTURAS = ["Lúpulo", "Pistache", "Avelã", "Amêndoa", "Açafrão", "Trufas"]
LIMIAR = 0.5

# Mapa de classes: atual / emergente 2050 / emergente 2080 / sempre inapta
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

# Cores: 0=inapta(branco), 1=apta atual(verde), 2=emergente 2050(azul claro), 3=emergente 2080(azul escuro)
cmap_class = mcolors.ListedColormap(['#f5f5f5', '#2d7d46', '#74b9e8', '#1a5fa8'])
norm_class  = mcolors.BoundaryNorm([0, 1, 2, 3, 4], 4)

labels = ['Inapta', 'Apta atual', 'Emergente 2050', 'Emergente 2080']
cores  = ['#f5f5f5', '#2d7d46', '#74b9e8', '#1a5fa8']

for i, cultura in enumerate(CULTURAS):
    ax = axes[i]

    atual    = ds_apt_atual[cultura].values
    fut_2050 = np.maximum(ds_fut_245_2050[cultura].values,
                          ds_fut_585_2050[cultura].values)  # mais otimista dos dois SSPs
    fut_2080 = np.maximum(ds_fut_245_2080[cultura].values,
                          ds_fut_585_2080[cultura].values)

    # Classificação
    classe = np.zeros_like(atual)
    classe[atual > LIMIAR]                                     = 1  # apta atual
    classe[(atual <= LIMIAR) & (fut_2050 > LIMIAR)]           = 2  # emergente 2050
    classe[(atual <= LIMIAR) & (fut_2050 <= LIMIAR) & (fut_2080 > LIMIAR)] = 3  # emergente 2080
    classe[np.isnan(atual)]                                    = np.nan

    im = ax.pcolormesh(lon, lat, classe, cmap=cmap_class, norm=norm_class, rasterized=True)
    estados.boundary.plot(ax=ax, linewidth=0.4, color='black', alpha=0.7)

    # Contar pixels de cada classe
    n1 = int(np.nansum(classe == 1))
    n2 = int(np.nansum(classe == 2))
    n3 = int(np.nansum(classe == 3))

    ax.set_title(f"{cultura}", fontsize=13, fontweight='bold')
    ax.set_xlim(-74, -28); ax.set_ylim(-34, 6)
    ax.set_xlabel('Longitude', fontsize=9)
    ax.set_ylabel('Latitude', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.text(0.02, 0.04,
            f"Atual: {n1}px | Emerg.2050: {n2}px | Emerg.2080: {n3}px",
            transform=ax.transAxes, fontsize=7,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

# Legenda global
patches = [mpatches.Patch(color=c, label=l) for c, l in zip(cores[1:], labels[1:])]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.02), frameon=True)

plt.suptitle('Regiões Aptas e Emergentes por Cultura — Brasil\n(envelope SSP2-4.5 e SSP5-8.5)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f"{DIR_ERA5}/fig3_regioes_emergentes.png", dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✓ Figura 3 salva!")

In [ ]:
# Rodar no Colab para pegar os números exatos
import numpy as np
import xarray as xr

DIR_ERA5 = "/content/drive/MyDrive/era5_brasil"

ds_atual = xr.open_dataset(f"{DIR_ERA5}/aptidao_atual_brasil.nc", engine="netcdf4")
ds_585_2080 = xr.open_dataset(f"{DIR_ERA5}/aptidao_ssp585_2071_2090_v2.nc", engine="netcdf4")

# Área total emergente por cultura (pixels únicos)
area_px_ha = 10978  # ha por pixel a 0.1° (~27°S)

print("=== NÚMEROS PARA O ABSTRACT ===")
total_emergente_ha = 0
for cultura in ["Lúpulo","Pistache","Avelã","Amêndoa","Açafrão","Trufas"]:
    atual = ds_atual[cultura].values
    fut   = ds_585_2080[cultura].values
    emergente = np.nansum((atual <= 0.5) & (fut > 0.5))
    ha = emergente * area_px_ha / 1e6
    total_emergente_ha += emergente * area_px_ha
    print(f"  {cultura}: {emergente} pixels emergentes = {ha:.1f} Mha")

print(f"\nTotal emergente (SSP585-2080): {total_emergente_ha/1e6:.0f} Mha")
print(f"Importação total 2015-2024: US$ {(431+306+232+57+2.5)/1e3:.1f} bilhões")

In [ ]:
import os

# Procurar os arquivos MapBiomas no Drive montado
print("Procurando arquivos MapBiomas no Drive...")
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if "mapbiomas" in f.lower() or "agriculture" in f.lower() or "coverage" in f.lower() or "lclu" in f.lower():
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1e9
            print(f"  {path} — {size:.2f} GB")

In [ ]:
import rasterio
import numpy as np

ARQUIVO_COVERAGE = "/content/drive/MyDrive/era5_brasil/2023_coverage_lclu_1-1-1_2242342e-4c61-44e4-83ac-2a2e7e23b148.tif"
ARQUIVO_AGRI     = "/content/drive/MyDrive/era5_brasil/2023_agriculture_agricultural_use_1-1-1_fb481488-0600-4b1b-9cbd-123b050afe40.tif"

for nome, path in [("COVERAGE", ARQUIVO_COVERAGE), ("AGRICULTURE", ARQUIVO_AGRI)]:
    print(f"\n=== {nome} ===")
    with rasterio.open(path) as src:
        print(f"CRS:       {src.crs}")
        print(f"Resolução: {src.res}")
        print(f"Dimensões: {src.width} x {src.height}")
        print(f"Extent:    {src.bounds}")
        print(f"NoData:    {src.nodata}")

        # Ler amostra reduzida para ver classes (evita carregar 1 GB inteiro)
        data = src.read(1, out_shape=(2000, 2000))
        classes = sorted([int(c) for c in np.unique(data) if c != src.nodata and c != 0])
        print(f"Classes:   {classes}")

In [ ]:
!pip install xarray netCDF4 rasterio geopandas scipy matplotlib -q

import numpy as np
import xarray as xr
import rasterio
from rasterio.windows import from_bounds
from scipy import stats as sp_stats
import os

MAPBIOMAS_TIF = "/content/drive/MyDrive/era5_brasil/2023_coverage_lclu_1-1-1_2242342e-4c61-44e4-83ac-2a2e7e23b148.tif"
DATA_DIR = "/content/drive/MyDrive/era5_brasil"

CLASSES_AGRICOLAS = {9,15,20,21,25,35,39,40,41,46,47,48,62,75}

ds_clim  = xr.open_dataset(f"{DATA_DIR}/climatologia_completa_brasil.nc", engine="netcdf4")
lat_era5 = ds_clim["lat"].values
lon_era5 = ds_clim["lon"].values
res      = abs(lat_era5[1] - lat_era5[0])

print(f"Grid ERA5: {len(lat_era5)} x {len(lon_era5)} pixels")
print("Iniciando reamostragem MapBiomas → ERA5 (pode demorar 15-30 min)...")

classes_era5 = np.zeros((len(lat_era5), len(lon_era5)), dtype=np.int16)
mascara      = np.zeros((len(lat_era5), len(lon_era5)), dtype=np.uint8)

with rasterio.open(MAPBIOMAS_TIF) as src:
    nodata = src.nodata or 0
    for i, lat in enumerate(lat_era5):
        if i % 20 == 0:
            print(f"  {i}/{len(lat_era5)} linhas ({i/len(lat_era5)*100:.0f}%)...", end="\r")
        for j, lon in enumerate(lon_era5):
            try:
                window = from_bounds(
                    lon - res/2, lat - res/2,
                    lon + res/2, lat + res/2,
                    src.transform
                )
                data = src.read(1, window=window)
                data = data[(data != nodata) & (data > 0)]
                if len(data) == 0:
                    continue
                moda = int(sp_stats.mode(data, keepdims=True).mode[0])
                classes_era5[i, j] = moda
                mascara[i, j] = 1 if moda in CLASSES_AGRICOLAS else 0
            except:
                continue

print(f"\n✓ Concluído!")
print(f"Pixels agrícolas: {int(mascara.sum()):,}")

# Salvar máscara
coords = {"lat": lat_era5, "lon": lon_era5}
ds_mask = xr.Dataset({
    "mascara": xr.DataArray(mascara, dims=["lat","lon"], coords=coords),
    "classe":  xr.DataArray(classes_era5, dims=["lat","lon"], coords=coords),
})
ds_mask.to_netcdf(f"{DATA_DIR}/mapbiomas_mask_era5.nc")
print(f"✓ Salvo: mapbiomas_mask_era5.nc")

In [ ]:
# Ver distribuição das classes encontradas
import numpy as np

CLASSES_AGRICOLAS_NOMES = {
    9:"Silvicultura", 15:"Pastagem", 20:"Cana-de-acucar",
    21:"Mosaico de Usos", 25:"Area nao vegetada", 35:"Dende",
    39:"Soja", 40:"Arroz", 41:"Outras Lavouras Temp",
    46:"Cafe", 47:"Citrus", 48:"Outras Lavouras Per",
    62:"Algodao", 75:"Outras Lavouras Temp 2"
}

EXCLUIDAS_NOMES = {
    3:"Formacao Florestal", 4:"Savana Florestal", 5:"Mangue",
    6:"Floresta Alagada", 11:"Campo Alagado", 12:"Campo nativo",
    23:"Praia/Duna", 24:"Area Urbana", 29:"Afloramento Rochoso",
    30:"Mineracao", 31:"Aquicultura", 32:"Salgado",
    33:"Rio/Lago/Oceano", 49:"Restinga Arborea", 50:"Restinga Herbacea"
}

classes_unicas, contagens = np.unique(
    classes_era5[classes_era5 > 0], return_counts=True
)

print("Distribuição das classes no grid ERA5 (0,1°):\n")
print(f"{'Status':<3} {'Cod':>4} {'Nome':<35} {'Pixels':>7} {'%Brasil':>8}")
print("-" * 62)

total_br = int((classes_era5 > 0).sum())
for cls, cnt in sorted(zip(classes_unicas, contagens), key=lambda x: -x[1]):
    cls = int(cls)
    if cls in CLASSES_AGRICOLAS_NOMES:
        status = "✓"
        nome   = CLASSES_AGRICOLAS_NOMES[cls]
    elif cls in EXCLUIDAS_NOMES:
        status = "✗"
        nome   = EXCLUIDAS_NOMES[cls]
    else:
        status = "?"
        nome   = f"Classe {cls}"
    print(f"  {status}  {cls:4d}  {nome:<35} {cnt:7,}  {cnt/total_br*100:7.1f}%")

print(f"\nTotal pixels com classe: {total_br:,}")
print(f"Pixels agrícolas (✓):    {int(mascara.sum()):,} ({int(mascara.sum())/total_br*100:.1f}%)")
print(f"Pixels excluídos (✗):    {total_br - int(mascara.sum()):,} ({(total_br-int(mascara.sum()))/total_br*100:.1f}%)")

In [ ]:
import xarray as xr
import numpy as np

DATA_DIR = "/content/drive/MyDrive/era5_brasil"

CULTURAS = ["Lúpulo","Pistache","Avelã","Amêndoa","Açafrão","Trufas"]
LIMIAR   = 0.5

ds_atual = xr.open_dataset(f"{DATA_DIR}/aptidao_atual_brasil.nc", engine="netcdf4")
lat      = ds_atual["lat"].values
lon      = ds_atual["lon"].values

# Carregar aptidão futura dos 4 cenários
cenarios = {}
for ssp in ["ssp245","ssp585"]:
    for periodo in ["2041_2060","2071_2090"]:
        key = f"{ssp}_{periodo}"
        cenarios[key] = xr.open_dataset(f"{DATA_DIR}/aptidao_{key}_v2.nc", engine="netcdf4")

print("Calculando zonas emergentes...")
emergentes = {}
for c in CULTURAS:
    apt_atual = ds_atual[c].values

    # Envelope dos dois SSPs por período
    apt_2050 = np.maximum(cenarios["ssp245_2041_2060"][c].values,
                          cenarios["ssp585_2041_2060"][c].values)
    apt_2080 = np.maximum(cenarios["ssp245_2071_2090"][c].values,
                          cenarios["ssp585_2071_2090"][c].values)

    classe = np.full_like(apt_atual, np.nan)
    classe[apt_atual > LIMIAR]                                                     = 1
    classe[(apt_atual <= LIMIAR) & (apt_2050 > LIMIAR)]                           = 2
    classe[(apt_atual <= LIMIAR) & (apt_2050 <= LIMIAR) & (apt_2080 > LIMIAR)]   = 3
    classe[np.isnan(apt_atual)]                                                    = np.nan

    emergentes[c] = classe
    n1,n2,n3 = int(np.nansum(classe==1)), int(np.nansum(classe==2)), int(np.nansum(classe==3))
    print(f"  {c:<10}: Apto={n1}px | Emerg.2050={n2}px | Emerg.2080={n3}px")

ds_emerg = xr.Dataset({
    c: xr.DataArray(v, dims=["lat","lon"], coords={"lat":lat,"lon":lon},
                    attrs={"classes":"1=apto_atual 2=emergente_2050 3=emergente_2080"})
    for c, v in emergentes.items()
})
ds_emerg.to_netcdf(f"{DATA_DIR}/zonas_emergentes.nc", mode="w")
print(f"\n✓ Salvo: {DATA_DIR}/zonas_emergentes.nc")

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import geopandas as gpd
import requests, zipfile, glob, os

DATA_DIR   = "/content/drive/MyDrive/era5_brasil"
OUTPUT_DIR = "/content/drive/MyDrive/era5_brasil/figuras_filtradas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CULTURAS      = ["Lúpulo","Pistache","Avelã","Amêndoa","Açafrão","Trufas"]
LIMIAR        = 0.5
AREA_PX_HA    = 10978
CENARIOS      = ["ssp245","ssp585"]
PERIODOS      = ["2041_2060","2071_2090"]
CLASSES_AGRI  = {9,15,20,21,25,35,39,40,41,46,47,48,62,75}

# Carregar máscara
ds_mask      = xr.open_dataset(f"{DATA_DIR}/mapbiomas_mask_era5.nc", engine="netcdf4")
mascara      = ds_mask["mascara"].values
lat_era5     = ds_mask["lat"].values
lon_era5     = ds_mask["lon"].values
mascara_br   = ~np.isnan(xr.open_dataset(
    f"{DATA_DIR}/climatologia_completa_brasil.nc",
    engine="netcdf4", decode_timedelta=False)["tmean"].mean("month").values)

# ── Aplicar máscara em todos os resultados ────────────────────────────────────
def filtrar(ds):
    dados = {}
    for c in CULTURAS:
        if c not in ds: continue
        iac = ds[c].values.copy().astype(float)
        iac[mascara == 0] = np.nan
        dados[c] = xr.DataArray(iac, dims=["lat","lon"],
                                coords={"lat":lat_era5,"lon":lon_era5})
    return xr.Dataset(dados)

print("Aplicando máscara...")
ds_atual     = xr.open_dataset(f"{DATA_DIR}/aptidao_atual_brasil.nc",      engine="netcdf4")
ds_atual_f   = filtrar(ds_atual)
ds_atual_f.to_netcdf(f"{DATA_DIR}/aptidao_filtrada_atual.nc", mode="w")
print("✓ aptidao_filtrada_atual.nc")

for ssp in CENARIOS:
    for periodo in PERIODOS:
        key = f"{ssp}_{periodo}"
        ds_f = filtrar(xr.open_dataset(f"{DATA_DIR}/aptidao_{key}_v2.nc", engine="netcdf4"))
        ds_f.to_netcdf(f"{DATA_DIR}/aptidao_filtrada_{key}.nc", mode="w")
        print(f"✓ aptidao_filtrada_{key}.nc")

# Zonas emergentes filtradas
ds_emerg = xr.open_dataset(f"{DATA_DIR}/zonas_emergentes.nc", engine="netcdf4")
dados_em = {}
for c in CULTURAS:
    if c not in ds_emerg: continue
    cl = ds_emerg[c].values.copy().astype(float)
    cl[mascara == 0] = np.nan
    dados_em[c] = xr.DataArray(cl, dims=["lat","lon"],
                                coords={"lat":lat_era5,"lon":lon_era5})
ds_emerg_f = xr.Dataset(dados_em)
ds_emerg_f.to_netcdf(f"{DATA_DIR}/zonas_emergentes_filtradas.nc", mode="w")
print("✓ zonas_emergentes_filtradas.nc")

# ── Tabela comparativa ────────────────────────────────────────────────────────
import pandas as pd
print("\n=== COMPARAÇÃO ANTES/DEPOIS DO FILTRO MAPBIOMAS ===")
print(f"\n{'Cultura':<12} {'Apto atual':>18} {'Emergente 2080':>18} {'Redução':>8}")
print(f"{'':12} {'Sem(Mha)':>9}{'Com(Mha)':>9} {'Sem(Mha)':>9}{'Com(Mha)':>9} {'apto':>8}")
print("-"*70)
rows = []
for c in CULTURAS:
    apt_s  = ds_atual[c].values
    apt_c  = ds_atual_f[c].values
    em_s   = ds_emerg[c].values
    em_c   = ds_emerg_f[c].values

    n_s  = int(np.nansum(apt_s[mascara_br] > LIMIAR))
    n_c  = int(np.nansum(apt_c[mascara_br] > LIMIAR))
    em_s_n = int(np.nansum((em_s==2)|(em_s==3)))
    em_c_n = int(np.nansum((em_c==2)|(em_c==3)))
    red  = (1 - n_c/max(n_s,1))*100

    mha_s  = n_s  * AREA_PX_HA / 1e6
    mha_c  = n_c  * AREA_PX_HA / 1e6
    emha_s = em_s_n * AREA_PX_HA / 1e6
    emha_c = em_c_n * AREA_PX_HA / 1e6

    print(f"  {c:<10} {mha_s:>8.1f} {mha_c:>8.1f}  {emha_s:>8.1f} {emha_c:>8.1f}  -{red:.0f}%")
    rows.append({"cultura":c,"apto_sem_Mha":round(mha_s,1),"apto_com_Mha":round(mha_c,1),
                 "emerg_sem_Mha":round(emha_s,1),"emerg_com_Mha":round(emha_c,1),
                 "reducao_pct":round(red,1)})

pd.DataFrame(rows).to_csv(f"{DATA_DIR}/area_comparison.csv", index=False)

# ── Shapefile estados ─────────────────────────────────────────────────────────
if not glob.glob("/tmp/estados_shp/*.shp"):
    r = requests.get("https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/Brasil/BR/BR_UF_2022.zip", stream=True)
    with open("/tmp/estados.zip","wb") as f:
        for chunk in r.iter_content(65536*16): f.write(chunk)
    os.makedirs("/tmp/estados_shp/", exist_ok=True)
    with zipfile.ZipFile("/tmp/estados.zip") as z: z.extractall("/tmp/estados_shp/")
estados = gpd.read_file(glob.glob("/tmp/estados_shp/*.shp")[0])

# ── Figura 1 filtrada ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
cmap = plt.cm.RdYlGn
norm = mcolors.Normalize(vmin=0, vmax=1)

for i, c in enumerate(CULTURAS):
    ax  = axes[i]
    apt = ds_atual_f[c].values
    im  = ax.pcolormesh(lon_era5, lat_era5, apt, cmap=cmap, norm=norm, rasterized=True)
    estados.boundary.plot(ax=ax, linewidth=0.4, color="black", alpha=0.7)
    ax.set_xlim(-74,-28); ax.set_ylim(-34,6)

    apt_v     = apt[mascara_br & ~np.isnan(apt)]
    n_ag      = int((mascara & mascara_br).sum())
    pct_apto  = np.sum(apt_v > LIMIAR) / n_ag * 100
    pct_otimo = np.sum(apt_v > 0.7)    / n_ag * 100

    ax.set_title(c, fontsize=13, fontweight="bold")
    ax.set_xlabel("Longitude",fontsize=9); ax.set_ylabel("Latitude",fontsize=9)
    ax.tick_params(labelsize=8)
    ax.text(0.02,0.04,f"Apta: {pct_apto:.1f}% | Ótima: {pct_otimo:.1f}%\n(filtro MapBiomas 2023)",
            transform=ax.transAxes, fontsize=7.5,
            bbox=dict(boxstyle="round,pad=0.3",facecolor="white",alpha=0.85))
    cb = plt.colorbar(im,ax=ax,shrink=0.8,pad=0.02)
    cb.set_label("Índice de Aptidão (0–1)",fontsize=8); cb.ax.tick_params(labelsize=7)

plt.suptitle("Aptidão Climática Atual — Brasil (1991–2020)\nFiltrado por uso do solo MapBiomas Coleção 9 (2023)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_aptidao_filtrada.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ fig1_aptidao_filtrada.png")

# ── Figura 3 filtrada ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
cores  = ["#f5f5f5","#2d7d46","#74b9e8","#1a5fa8"]
labels = ["Inapta/Excluída","Apta atual","Emergente 2050","Emergente 2080"]
cmap_e = mcolors.ListedColormap(cores)
norm_e = mcolors.BoundaryNorm([0,1,2,3,4],4)

for i, c in enumerate(CULTURAS):
    ax = axes[i]
    cl = ds_emerg_f[c].values
    ax.pcolormesh(lon_era5, lat_era5, cl, cmap=cmap_e, norm=norm_e, rasterized=True)
    estados.boundary.plot(ax=ax, linewidth=0.4, color="black", alpha=0.7)
    ax.set_xlim(-74,-28); ax.set_ylim(-34,6)

    n1,n2,n3 = int(np.nansum(cl==1)), int(np.nansum(cl==2)), int(np.nansum(cl==3))
    ax.set_title(c, fontsize=13, fontweight="bold")
    ax.set_xlabel("Longitude",fontsize=9); ax.set_ylabel("Latitude",fontsize=9)
    ax.tick_params(labelsize=8)
    ax.text(0.02,0.04,
            f"Atual:{n1*AREA_PX_HA/1e6:.1f}Mha | Em.50:{n2*AREA_PX_HA/1e6:.1f}Mha | Em.80:{n3*AREA_PX_HA/1e6:.1f}Mha\n(filtro MapBiomas 2023)",
            transform=ax.transAxes, fontsize=7,
            bbox=dict(boxstyle="round,pad=0.3",facecolor="white",alpha=0.85))

patches = [mpatches.Patch(color=c,label=l) for c,l in zip(cores[1:],labels[1:])]
fig.legend(handles=patches,loc="lower center",ncol=3,fontsize=11,bbox_to_anchor=(0.5,-0.02),frameon=True)
plt.suptitle("Regiões Aptas e Emergentes — Brasil\n(filtro MapBiomas Coleção 9 2023 | envelope SSP2-4.5 e SSP5-8.5)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_emergentes_filtradas.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ fig3_emergentes_filtradas.png")

In [ ]:
!pip install netCDF4 -q
import IPython
IPython.Application.instance().kernel.do_shutdown(True)